<a href="https://colab.research.google.com/github/nehansa2003/Weather-Forecasting-and-Analysis-Using-LSTM/blob/main/notebooks/1_Capstone_Data_Scrape1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Sri Lanka District Weather Scraper (Open-Meteo Historical Archive API)
========================================================================
Scrapes hourly historical weather data (2020-01-01 to 2025-12-31) for
multiple sample points inside every Sri Lankan district. The number of
sample points per district scales with the district's surface area.

NO API KEY REQUIRED — Open-Meteo's archive API is free and open.

Output: one CSV file per district in OUTPUT_DIR, containing hourly rows
for every sample point in that district, with columns for location,
lat/lon/elevation, time fields, and all weather variables.

Dependencies:
    pip install requests pandas numpy
    (optional, for real polygon-based sampling) pip install geopandas shapely
"""

"\nSri Lanka District Weather Scraper (Open-Meteo Historical Archive API)\n========================================================================\nScrapes hourly historical weather data (2020-01-01 to 2025-12-31) for\nmultiple sample points inside every Sri Lankan district. The number of\nsample points per district scales with the district's surface area.\n\nNO API KEY REQUIRED — Open-Meteo's archive API is free and open.\n\nOutput: one CSV file per district in OUTPUT_DIR, containing hourly rows\nfor every sample point in that district, with columns for location,\nlat/lon/elevation, time fields, and all weather variables.\n\nDependencies:\n    pip install requests pandas numpy\n    (optional, for real polygon-based sampling) pip install geopandas shapely\n"

In [ ]:
import os
import math
import time
import random
import logging
from datetime import date, timedelta

import numpy as np
import pandas as pd
import requests


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────

START_YEAR = 2020
END_YEAR = 2025

# Approx. km^2 of district area per sample point. Lower = more points.
POINT_DENSITY_KM2 = 100

# Minimum number of sample points per district, regardless of area.
MIN_POINTS_PER_DISTRICT = 1

OUTPUT_DIR = "sl_weather_data"
REQUEST_TIMEOUT = 30
MAX_RETRIES = 5
BASE_SLEEP = 1.0  # seconds, used for exponential backoff between retries

BASE_URL = "https://archive-api.open-meteo.com/v1/archive"


In [ ]:
# Optional: path to a district-level polygon file (GeoJSON/Shapefile) with a
# "district" name column. If provided and geopandas/shapely are installed,
# real polygon-based sampling is used instead of the circular approximation.
DISTRICT_POLYGONS_PATH = None  # e.g. "lk_districts.geojson"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
log = logging.getLogger(__name__)


In [ ]:
# ─────────────────────────────────────────────────────────────────
# SRI LANKA DISTRICTS — centroid (lat, lon) and approximate area (km^2)
# Areas are approximate published figures; centroids are approximate
# geographic centers. Good enough for grid sampling; replace with real
# polygons via DISTRICT_POLYGONS_PATH for higher accuracy.
# ─────────────────────────────────────────────────────────────────

DISTRICTS = {
    "Colombo":       {"lat": 6.9271, "lon": 79.8612, "area_km2": 699},
    "Gampaha":       {"lat": 7.0917, "lon": 80.0000, "area_km2": 1391},
    "Kalutara":      {"lat": 6.5854, "lon": 80.1000, "area_km2": 1598},
    "Kandy":         {"lat": 7.2906, "lon": 80.6337, "area_km2": 1940},
    "Matale":        {"lat": 7.4675, "lon": 80.6234, "area_km2": 1993},
    "Nuwara Eliya":  {"lat": 6.9497, "lon": 80.7891, "area_km2": 1741},
    "Galle":         {"lat": 6.0535, "lon": 80.2210, "area_km2": 1652},
    "Matara":        {"lat": 5.9549, "lon": 80.5550, "area_km2": 1283},
    "Hambantota":    {"lat": 6.1241, "lon": 81.1185, "area_km2": 2609},
    #"Jaffna":        {"lat": 9.6615, "lon": 80.0255, "area_km2": 1025},
    #"Kilinochchi":   {"lat": 9.3961, "lon": 80.3982, "area_km2": 1279},
    #"Mannar":        {"lat": 8.9810, "lon": 79.9044, "area_km2": 1996},
    #"Vavuniya":      {"lat": 8.7514, "lon": 80.4971, "area_km2": 1967},
    #"Mullaitivu":    {"lat": 9.2670, "lon": 80.8142, "area_km2": 2617},
    "Batticaloa":    {"lat": 7.7170, "lon": 81.7000, "area_km2": 2854},
    "Ampara":        {"lat": 7.2833, "lon": 81.6667, "area_km2": 4415},
    "Trincomalee":   {"lat": 8.5874, "lon": 81.2152, "area_km2": 2727},
    #"Kurunegala":    {"lat": 7.4863, "lon": 80.3647, "area_km2": 4816},
    #"Puttalam":      {"lat": 8.0362, "lon": 79.8283, "area_km2": 3072},
    "Anuradhapura":  {"lat": 8.3114, "lon": 80.4037, "area_km2": 7179},
    "Polonnaruwa":   {"lat": 7.9403, "lon": 81.0188, "area_km2": 3293},
    #"Badulla":       {"lat": 6.9934, "lon": 81.0550, "area_km2": 2861},
    #"Moneragala":    {"lat": 6.8721, "lon": 81.3507, "area_km2": 5639},
    #"Ratnapura":     {"lat": 6.6828, "lon": 80.4126, "area_km2": 3275},
    #"Kegalle":       {"lat": 7.2513, "lon": 80.3464, "area_km2": 1693},
}

In [ ]:
HOURLY_VARS = [
    "temperature_2m",
    "apparent_temperature",
    "relative_humidity_2m",
    "dew_point_2m",
    "pressure_msl",
    "wind_speed_10m",
    "wind_direction_10m",
    "wind_gusts_10m",
    "precipitation",
    "rain",
    "cloud_cover",
    "shortwave_radiation",
    "sunshine_duration",
    "visibility",
]

In [ ]:
# ─────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────

def deg_to_compass(deg):
    """Convert a wind direction in degrees to a 16-point compass label."""
    if deg is None or (isinstance(deg, float) and math.isnan(deg)):
        return None
    dirs = [
        "N", "NNE", "NE", "ENE", "E", "ESE", "SE", "SSE",
        "S", "SSW", "SW", "WSW", "W", "WNW", "NW", "NNW",
    ]
    ix = int((deg / 22.5) + 0.5) % 16
    return dirs[ix]

def get_season(month):
    """Approximate Sri Lankan monsoon-based seasons by month."""
    if month in (12, 1, 2):
        return "Northeast_Monsoon"
    if month in (3, 4):
        return "First_Intermonsoon"
    if month in (5, 6, 7, 8, 9):
        return "Southwest_Monsoon"
    if month in (10, 11):
        return "Second_Intermonsoon"
    return "Unknown"


def year_chunks(start_year, end_year):
    """Yield (start_date, end_date) strings, one per calendar year, capped
    so we never request a date range beyond 'today'."""
    today = date.today()
    for yr in range(start_year, end_year + 1):
        start = date(yr, 1, 1)
        end = date(yr, 12, 31)
        if start > today:
            break
        if end > today:
            end = today
        yield start.isoformat(), end.isoformat()

In [ ]:

# DISTRICT GRID CONFIGURATION

DISTRICT_POLYGONS_PATH = None

In [ ]:
# ─────────────────────────────────────────────────────────────────
# GRID GENERATION
# ─────────────────────────────────────────────────────────────────

POINT_AREA_KM2 = 50.0


def num_points_for_area(
    area_km2,
    area_per_point_km2=POINT_AREA_KM2
):
    """
    Calculate the number of sampling points for a district.

    Rule:
        1 sampling point per 50 km²

    ceil() is used so that the entire district area is represented.

    Examples:
        699 km²  -> 14 points
        1391 km² -> 28 points
        7179 km² -> 144 points
    """

    if area_km2 <= 0:
        return 1

    return max(
        1,
        math.ceil(
            area_km2 / area_per_point_km2
        )
    )


# ─────────────────────────────────────────────────────────────────
# CIRCULAR GRID
# ─────────────────────────────────────────────────────────────────

def generate_grid_points_circular(
    center_lat,
    center_lon,
    area_km2,
    area_per_point_km2=POINT_AREA_KM2
):
    """
    Generate sampling points for a district.

    The district is approximated as a circle having the same
    area as the actual district.

    Number of points:

        ceil(area_km2 / 50)

    Points are distributed on a regular square grid and only
    points falling inside the circular approximation are retained.

    Returns:
        List of (latitude, longitude)
    """

    # -------------------------------------------------------------
    # Number of required points
    # -------------------------------------------------------------

    n_target = num_points_for_area(
        area_km2,
        area_per_point_km2
    )

    # -------------------------------------------------------------
    # Equivalent circular radius
    # -------------------------------------------------------------

    radius_km = math.sqrt(
        area_km2 / math.pi
    )

    # -------------------------------------------------------------
    # Convert km to degrees
    # -------------------------------------------------------------

    km_per_deg_lat = 111.0

    km_per_deg_lon = (
        111.0 *
        math.cos(
            math.radians(center_lat)
        )
    )

    radius_deg_lat = (
        radius_km /
        km_per_deg_lat
    )

    radius_deg_lon = (
        radius_km /
        km_per_deg_lon
    )

    # -------------------------------------------------------------
    # If only one point is required
    # -------------------------------------------------------------

    if n_target <= 1:

        return [
            (
                round(center_lat, 5),
                round(center_lon, 5)
            )
        ]

    # -------------------------------------------------------------
    # Generate a sufficiently dense grid.
    #
    # We generate MORE candidate points than required and then
    # select approximately n_target points.
    # -------------------------------------------------------------

    grid_side = max(
        2,
        math.ceil(
            math.sqrt(
                n_target /
                (math.pi / 4)
            )
        )
    )

    # -------------------------------------------------------------
    # Generate latitude/longitude positions
    # -------------------------------------------------------------

    lat_steps = np.linspace(
        -radius_deg_lat,
        radius_deg_lat,
        grid_side
    )

    lon_steps = np.linspace(
        -radius_deg_lon,
        radius_deg_lon,
        grid_side
    )

    candidates = []

    for dlat in lat_steps:

        for dlon in lon_steps:

            # Distance from center in km
            dist_km = math.sqrt(

                (dlat * km_per_deg_lat) ** 2

                +

                (dlon * km_per_deg_lon) ** 2
            )

            # Keep points inside circle
            if dist_km <= radius_km:

                lat = round(
                    center_lat + dlat,
                    5
                )

                lon = round(
                    center_lon + dlon,
                    5
                )

                candidates.append(
                    (
                        lat,
                        lon
                    )
                )

    # -------------------------------------------------------------
    # Safety check
    # -------------------------------------------------------------

    if not candidates:

        return [
            (
                round(center_lat, 5),
                round(center_lon, 5)
            )
        ]

    # -------------------------------------------------------------
    # If candidate count is greater than target,
    # select approximately n_target evenly distributed points.
    # -------------------------------------------------------------

    if len(candidates) > n_target:

        indices = np.linspace(
            0,
            len(candidates) - 1,
            n_target,
            dtype=int
        )

        points = [
            candidates[i]
            for i in indices
        ]

    else:

        points = candidates

    # -------------------------------------------------------------
    # Always make sure center point exists when possible.
    # -------------------------------------------------------------

    center_point = (
        round(center_lat, 5),
        round(center_lon, 5)
    )

    if center_point not in points:

        if len(points) >= n_target:

            # Replace the point furthest from center
            max_index = max(

                range(len(points)),

                key=lambda i: math.sqrt(

                    (
                        (
                            points[i][0]
                            -
                            center_lat
                        )
                        *
                        km_per_deg_lat
                    ) ** 2

                    +

                    (
                        (
                            points[i][1]
                            -
                            center_lon
                        )
                        *
                        km_per_deg_lon
                    ) ** 2
                )
            )

            points[max_index] = center_point

        else:

            points.append(
                center_point
            )

    return points


# ─────────────────────────────────────────────────────────────────
# POLYGON GRID
# ─────────────────────────────────────────────────────────────────

def generate_grid_points_from_polygon(
    polygon,
    area_per_point_km2=POINT_AREA_KM2
):
    """
    Generate sampling points inside a real district polygon.

    The polygon must use geographic coordinates (longitude,
    latitude).

    Number of target points:

        ceil(district_area / 50 km²)

    """

    from shapely.geometry import Point

    # -------------------------------------------------------------
    # Polygon bounds
    # -------------------------------------------------------------

    minx, miny, maxx, maxy = (
        polygon.bounds
    )

    # -------------------------------------------------------------
    # Approximate polygon area in km²
    #
    # This approximation is acceptable for determining the number
    # of sampling points, but a projected CRS is preferred for
    # high-accuracy area calculations.
    # -------------------------------------------------------------

    center_lat = polygon.centroid.y

    km_per_deg_lat = 111.0

    km_per_deg_lon = (
        111.0 *
        math.cos(
            math.radians(center_lat)
        )
    )

    area_km2_est = (
        polygon.area
        *
        km_per_deg_lat
        *
        km_per_deg_lon
    )

    # -------------------------------------------------------------
    # Number of required points
    # -------------------------------------------------------------

    n_target = num_points_for_area(
        area_km2_est,
        area_per_point_km2
    )

    # -------------------------------------------------------------
    # Determine grid resolution
    # -------------------------------------------------------------

    grid_side = max(
        2,
        math.ceil(
            math.sqrt(
                n_target
            )
        )
    )

    # -------------------------------------------------------------
    # Generate grid
    # -------------------------------------------------------------

    xs = np.linspace(
        minx,
        maxx,
        grid_side
    )

    ys = np.linspace(
        miny,
        maxy,
        grid_side
    )

    candidates = []

    for x in xs:

        for y in ys:

            point = Point(
                x,
                y
            )

            if polygon.contains(
                point
            ):

                candidates.append(
                    (
                        round(y, 5),
                        round(x, 5)
                    )
                )

    # -------------------------------------------------------------
    # If no points found
    # -------------------------------------------------------------

    if not candidates:

        centroid = (
            polygon.centroid
        )

        return [
            (
                round(
                    centroid.y,
                    5
                ),

                round(
                    centroid.x,
                    5
                )
            )
        ]

    # -------------------------------------------------------------
    # Select target number
    # -------------------------------------------------------------

    if len(candidates) > n_target:

        indices = np.linspace(
            0,
            len(candidates) - 1,
            n_target,
            dtype=int
        )

        points = [
            candidates[i]
            for i in indices
        ]

    else:

        points = candidates

    return points


# ─────────────────────────────────────────────────────────────────
# LOAD DISTRICT POINT GRIDS
# ─────────────────────────────────────────────────────────────────

def load_district_point_grids():
    """
    Generate sampling points for every Sri Lankan district.

    Priority:

        1. Real district polygons if available
        2. Circular approximation otherwise

    Returns:

        {
            "Colombo": [
                (lat, lon),
                ...
            ],

            "Gampaha": [
                ...
            ],

            ...
        }
    """

    # =============================================================
    # OPTION 1 — REAL DISTRICT POLYGONS
    # =============================================================

    if (
        DISTRICT_POLYGONS_PATH
        and
        os.path.exists(
            DISTRICT_POLYGONS_PATH
        )
    ):

        try:

            import geopandas as gpd

            gdf = gpd.read_file(
                DISTRICT_POLYGONS_PATH
            )

            # -----------------------------------------------------
            # Find district name column
            # -----------------------------------------------------

            if "district" in gdf.columns:

                name_col = "district"

            elif "District" in gdf.columns:

                name_col = "District"

            elif "DISTRICT" in gdf.columns:

                name_col = "DISTRICT"

            else:

                name_col = gdf.columns[0]

            grids = {}

            # -----------------------------------------------------
            # Generate grid for each polygon
            # -----------------------------------------------------

            for _, row in gdf.iterrows():

                name = str(
                    row[name_col]
                ).strip()

                polygon = row.geometry

                if polygon is None:

                    continue

                points = (
                    generate_grid_points_from_polygon(
                        polygon,
                        POINT_AREA_KM2
                    )
                )

                grids[name] = points

                log.info(
                    "%s: %d sampling points "
                    "(polygon method)",
                    name,
                    len(points)
                )

            # -----------------------------------------------------
            # Make sure something was actually loaded
            # -----------------------------------------------------

            if grids:

                log.info(
                    "Loaded district polygons from %s",
                    DISTRICT_POLYGONS_PATH
                )

                return grids

            log.warning(
                "District polygon file was loaded, "
                "but no valid districts were found. "
                "Falling back to circular grids."
            )

        except ImportError:

            log.warning(
                "geopandas/shapely not installed. "
                "Using circular approximation."
            )

        except Exception as e:

            log.warning(
                "Could not load district polygons: %s. "
                "Using circular approximation.",
                e
            )

    # =============================================================
    # OPTION 2 — CIRCULAR APPROXIMATION
    # =============================================================

    grids = {}

    for name, info in DISTRICTS.items():

        points = (
            generate_grid_points_circular(

                center_lat=info["lat"],

                center_lon=info["lon"],

                area_km2=info["area_km2"],

                area_per_point_km2=POINT_AREA_KM2
            )
        )

        grids[name] = points

        log.info(
            "%s: %d sampling points "
            "(%.0f km² per point)",
            name,
            len(points),
            POINT_AREA_KM2
        )

    # =============================================================
    # VERY IMPORTANT
    #
    # This return statement fixes your current error.
    # =============================================================

    return grids

In [ ]:

# # GRID GENERATION

# # One sampling point for approximately every 50 km²
# AREA_PER_POINT_KM2 = 50.0


# def num_points_for_area(
#     area_km2,
#     area_per_point_km2=AREA_PER_POINT_KM2
# ):

#     if area_km2 <= 0:
#         return 1

#     return max(
#         1,
#         math.ceil(area_km2 / area_per_point_km2)
#     )


# # CIRCULAR GRID GENERATION

# def generate_grid_points_circular(
#     center_lat,
#     center_lon,
#     area_km2,
#     area_per_point_km2=AREA_PER_POINT_KM2
# ):


#     # Calculate number of required points

#     n_target = num_points_for_area(
#         area_km2,
#         area_per_point_km2
#     )


#     radius_km = math.sqrt(
#         area_km2 / math.pi
#     )

#     # Convert kilometres to latitude/longitude degrees

#     km_per_deg_lat = 111.0

#     km_per_deg_lon = (
#         111.0 *
#         math.cos(math.radians(center_lat))
#     )

#     radius_deg_lat = (
#         radius_km / km_per_deg_lat
#     )

#     radius_deg_lon = (
#         radius_km / km_per_deg_lon
#     )

#     # 4. If only one point is required

#     if n_target <= 1:

#         return [
#             (
#                 round(center_lat, 5),
#                 round(center_lon, 5)
#             )
#         ]


#     # Start with an estimated grid size
#     # A circle occupies approximately π/4 of its bounding square.
#     grid_side = max(
#         2,
#         math.ceil(
#             math.sqrt(
#                 n_target / (math.pi / 4)
#             )
#         )
#     )

#     points = []


#     # Keep increasing grid resolution until we have enough
#     # points inside the circular approximation.

#     max_iterations = 20

#     for _ in range(max_iterations):

#         lat_steps = np.linspace(
#             -radius_deg_lat,
#             radius_deg_lat,
#             grid_side
#         )

#         lon_steps = np.linspace(
#             -radius_deg_lon,
#             radius_deg_lon,
#             grid_side
#         )

#         points = []

#         # Generate grid points

#         for dlat in lat_steps:

#             for dlon in lon_steps:

#                 # Calculate distance from centre in kilometres

#                 dist_km = math.sqrt(
#                     (dlat * km_per_deg_lat) ** 2
#                     +
#                     (dlon * km_per_deg_lon) ** 2
#                 )

#                 # Keep only points inside the circle

#                 if dist_km <= radius_km:

#                     lat = round(
#                         center_lat + dlat,
#                         5
#                     )

#                     lon = round(
#                         center_lon + dlon,
#                         5
#                     )

#                     points.append(
#                         (lat, lon)
#                     )


#         # Check whether enough points were generated

#         if len(points) >= n_target:
#             break

#         # Increase grid resolution

#         grid_side += 1


#     #Safety fallback

#     if not points:

#         points = [
#             (
#                 round(center_lat, 5),
#                 round(center_lon, 5)
#             )
#         ]


#     # If more points than required were generated,
#     # select evenly distributed points.

#     if len(points) > n_target:

#         indices = np.linspace(
#             0,
#             len(points) - 1,
#             n_target,
#             dtype=int
#         )

#         points = [
#             points[i]
#             for i in indices
#         ]

#     return points



# # POLYGON GRID GENERATION

# def generate_grid_points_from_polygon(
#     polygon,
#     area_per_point_km2=AREA_PER_POINT_KM2,
#     min_points=1
# ):

#     from shapely.geometry import Point

#     # Polygon bounding box

#     minx, miny, maxx, maxy = polygon.bounds


#     # Estimate polygon area in km²
#     # Assumes polygon coordinates are longitude/latitude.

#     area_km2_est = polygon.area * (111.0 ** 2)

#     # Calculate required number of points

#     n_target = num_points_for_area(
#         area_km2_est,
#         area_per_point_km2
#     )

#     n_target = max(
#         min_points,
#         n_target
#     )


#     # Start with an initial grid resolution

#     grid_side = max(
#         2,
#         math.ceil(
#             math.sqrt(n_target * 2)
#         )
#     )

#     points = []

#     # Increase resolution until enough points are found

#     max_iterations = 20

#     for _ in range(max_iterations):

#         xs = np.linspace(
#             minx,
#             maxx,
#             grid_side
#         )

#         ys = np.linspace(
#             miny,
#             maxy,
#             grid_side
#         )

#         points = []

#         for x in xs:

#             for y in ys:

#                 if polygon.contains(
#                     Point(x, y)
#                 ):

#                     points.append(
#                         (
#                             round(y, 5),
#                             round(x, 5)
#                         )
#                     )

#         if len(points) >= n_target:
#             break

#         grid_side += 1

#     # Safety fallback

#     if not points:

#         centroid = polygon.centroid

#         points = [
#             (
#                 round(centroid.y, 5),
#                 round(centroid.x, 5)
#             )
#         ]


#     # Keep exactly n_target points
#     # when more points were generated.


#     if len(points) > n_target:

#         indices = np.linspace(
#             0,
#             len(points) - 1,
#             n_target,
#             dtype=int
#         )

#         points = [
#             points[i]
#             for i in indices
#         ]

#     return points



# # LOAD DISTRICT POINT GRIDS

# def load_district_point_grids():
#     """
#     Generate sampling grids for all Sri Lankan districts.

#     Sampling rule:
#         1 sampling point per 50 km².

#     If a real district polygon file is available, polygon-based
#     sampling is used.

#     Otherwise, the district is approximated as a circle using:

#         - district centroid
#         - district area

#     Returns:

#         {
#             "Colombo": [(lat, lon), ...],
#             "Gampaha": [(lat, lon), ...],
#             ...
#         }
#     """

#     # OPTION 1
#     # USE REAL DISTRICT POLYGONS

#     if (
#         DISTRICT_POLYGONS_PATH
#         and os.path.exists(
#             DISTRICT_POLYGONS_PATH
#         )
#     ):

#         try:

#             import geopandas as gpd

#             gdf = gpd.read_file(
#                 DISTRICT_POLYGONS_PATH
#             )

#             # Find district-name column

#             name_col = (
#                 "district"
#                 if "district" in gdf.columns
#                 else gdf.columns[0]
#             )

#             grids = {}

#             for _, row in gdf.iterrows():

#                 district_name = row[name_col]

#                 grids[district_name] = (
#                     generate_grid_points_from_polygon(
#                         row.geometry,
#                         area_per_point_km2=AREA_PER_POINT_KM2
#                     )
#                 )

#             log.info(
#                 "Loaded district polygons from %s",
#                 DISTRICT_POLYGONS_PATH
#             )

#             return grids

#         except ImportError:

#             log.warning(
#                 "geopandas/shapely not installed — "
#                 "falling back to circular approximation grid."
#             )

#         except Exception as e:

#             log.warning(
#                 "Failed to load district polygons: %s. "
#                 "Falling back to circular approximation grid.",
#                 e
#             )

In [ ]:
# # ─────────────────────────────────────────────────────────────────
# # HOURLY SCRAPING
# # ─────────────────────────────────────────────────────────────────

# # Number of geographic points included in one API request.
# #
# # You can reduce this to 10 if you still experience 429 errors.
# BATCH_SIZE = 10


# # Delay between successful API requests.
# #
# # Since you previously received many 429 errors, use a conservative
# # delay. You can later increase/decrease this.
# REQUEST_DELAY = 5.0


# # Maximum number of retries for one failed request.
# MAX_RETRIES = 5


# # Initial retry delay.
# BASE_SLEEP = 10.0


# # Maximum time allowed for one HTTP request.
# REQUEST_TIMEOUT = 180


# # ─────────────────────────────────────────────────────────────────
# # HOURLY VARIABLES
# # ─────────────────────────────────────────────────────────────────

# HOURLY_VARS = [

#     "temperature_2m",

#     "apparent_temperature",

#     "relative_humidity_2m",

#     "dew_point_2m",

#     "pressure_msl",

#     "wind_speed_10m",

#     "wind_direction_10m",

#     "wind_gusts_10m",

#     "precipitation",

#     "rain",

#     "cloud_cover",

#     "shortwave_radiation",

#     "sunshine_duration",

#     "visibility",
# ]


# # ─────────────────────────────────────────────────────────────────
# # REQUEST WITH RETRY
# # ─────────────────────────────────────────────────────────────────

# def fetch_with_retry(params):
#     """
#     Send one Open-Meteo API request with robust retry handling.

#     Handles:

#         429  -> rate limiting
#         500  -> server error
#         502  -> server error
#         503  -> server error
#         504  -> server error
#         timeout
#         connection errors

#     Returns:

#         JSON response
#         None if the request ultimately fails
#     """

#     for attempt in range(
#         1,
#         MAX_RETRIES + 1
#     ):

#         try:

#             # -----------------------------------------------------
#             # Send request
#             # -----------------------------------------------------

#             resp = requests.get(
#                 BASE_URL,
#                 params=params,
#                 timeout=REQUEST_TIMEOUT
#             )

#             # -----------------------------------------------------
#             # RATE LIMIT: 429
#             # -----------------------------------------------------

#             if resp.status_code == 429:

#                 retry_after = (
#                     resp.headers.get(
#                         "Retry-After"
#                     )
#                 )

#                 if retry_after:

#                     try:

#                         sleep_for = float(
#                             retry_after
#                         )

#                     except ValueError:

#                         sleep_for = (
#                             BASE_SLEEP *
#                             (2 ** (attempt - 1))
#                         )

#                 else:

#                     sleep_for = (
#                         BASE_SLEEP *
#                         (2 ** (attempt - 1))
#                     )

#                 # Random jitter prevents repeated requests from
#                 # occurring at exactly the same time.
#                 sleep_for += random.uniform(
#                     2,
#                     5
#                 )

#                 log.warning(
#                     "Rate limited (429). "
#                     "Attempt %d/%d. "
#                     "Waiting %.1f seconds...",
#                     attempt,
#                     MAX_RETRIES,
#                     sleep_for
#                 )

#                 time.sleep(
#                     sleep_for
#                 )

#                 continue

#             # -----------------------------------------------------
#             # TEMPORARY SERVER ERRORS
#             # -----------------------------------------------------

#             if resp.status_code in (
#                 500,
#                 502,
#                 503,
#                 504
#             ):

#                 sleep_for = (
#                     BASE_SLEEP *
#                     (2 ** (attempt - 1))
#                     +
#                     random.uniform(2, 5)
#                 )

#                 log.warning(
#                     "Server error %d. "
#                     "Attempt %d/%d. "
#                     "Waiting %.1f seconds...",
#                     resp.status_code,
#                     attempt,
#                     MAX_RETRIES,
#                     sleep_for
#                 )

#                 time.sleep(
#                     sleep_for
#                 )

#                 continue

#             # -----------------------------------------------------
#             # OTHER HTTP ERRORS
#             # -----------------------------------------------------

#             resp.raise_for_status()

#             # -----------------------------------------------------
#             # Successful response
#             # -----------------------------------------------------

#             data = resp.json()

#             # Small delay after successful request.
#             #
#             # This helps prevent the next request from immediately
#             # triggering a rate limit.
#             time.sleep(
#                 REQUEST_DELAY
#             )

#             return data

#         # ---------------------------------------------------------
#         # TIMEOUT
#         # ---------------------------------------------------------

#         except requests.exceptions.Timeout:

#             sleep_for = (
#                 BASE_SLEEP *
#                 (2 ** (attempt - 1))
#                 +
#                 random.uniform(2, 5)
#             )

#             log.warning(
#                 "Request timeout. "
#                 "Attempt %d/%d. "
#                 "Retrying in %.1f seconds...",
#                 attempt,
#                 MAX_RETRIES,
#                 sleep_for
#             )

#             time.sleep(
#                 sleep_for
#             )

#         # ---------------------------------------------------------
#         # CONNECTION ERROR
#         # ---------------------------------------------------------

#         except requests.exceptions.ConnectionError:

#             sleep_for = (
#                 BASE_SLEEP *
#                 (2 ** (attempt - 1))
#                 +
#                 random.uniform(2, 5)
#             )

#             log.warning(
#                 "Connection error. "
#                 "Attempt %d/%d. "
#                 "Retrying in %.1f seconds...",
#                 attempt,
#                 MAX_RETRIES,
#                 sleep_for
#             )

#             time.sleep(
#                 sleep_for
#             )

#         # ---------------------------------------------------------
#         # OTHER REQUEST ERROR
#         # ---------------------------------------------------------

#         except requests.exceptions.RequestException as e:

#             log.warning(
#                 "Request failed: %s",
#                 e
#             )

#             break

#         # ---------------------------------------------------------
#         # UNEXPECTED ERROR
#         # ---------------------------------------------------------

#         except Exception as e:

#             log.warning(
#                 "Unexpected error: %s",
#                 e
#             )

#             break

#     # -------------------------------------------------------------
#     # All attempts failed
#     # -------------------------------------------------------------

#     log.error(
#         "Giving up after %d attempts.",
#         MAX_RETRIES
#     )

#     return None


# # ─────────────────────────────────────────────────────────────────
# # BUILD DATAFRAME FOR ONE LOCATION
# # ─────────────────────────────────────────────────────────────────

# def build_point_dataframe(
#     district,
#     point_id,
#     lat,
#     lon,
#     data
# ):
#     """
#     Convert one Open-Meteo hourly response into a DataFrame.
#     """

#     if data is None:
#         return None

#     if "hourly" not in data:

#         log.warning(
#             "%s Point %s: "
#             "No hourly data returned.",
#             district,
#             point_id
#         )

#         return None

#     h = data["hourly"]

#     # -------------------------------------------------------------
#     # Elevation
#     # -------------------------------------------------------------

#     elevation = data.get(
#         "elevation"
#     )

#     # -------------------------------------------------------------
#     # Time
#     # -------------------------------------------------------------

#     times = pd.to_datetime(
#         h["time"]
#     )

#     # -------------------------------------------------------------
#     # Build dataframe
#     # -------------------------------------------------------------

#     df = pd.DataFrame({

#         "District":
#             district,

#         "PointID":
#             point_id,

#         "Latitude":
#             lat,

#         "Longitude":
#             lon,

#         "Elevation_m":
#             elevation,

#         "DateTime":
#             h["time"],

#         "Date":
#             times.date,

#         "Day":
#             times.day,

#         "Month":
#             times.month,

#         "Year":
#             times.year,

#         "Time":
#             times.strftime("%H:%M"),

#         "Season": [
#             get_season(month)
#             for month in times.month
#         ],

#         # ---------------------------------------------------------
#         # Temperature
#         # ---------------------------------------------------------

#         "Temperature_C":
#             h.get("temperature_2m"),

#         "FeelsLike_C":
#             h.get("apparent_temperature"),

#         # ---------------------------------------------------------
#         # Humidity
#         # ---------------------------------------------------------

#         "Humidity_pct":
#             h.get("relative_humidity_2m"),

#         "DewPoint_C":
#             h.get("dew_point_2m"),

#         # ---------------------------------------------------------
#         # Atmospheric pressure
#         # ---------------------------------------------------------

#         "Pressure_hPa":
#             h.get("pressure_msl"),

#         # ---------------------------------------------------------
#         # Wind
#         # ---------------------------------------------------------

#         "WindSpeed_kmh":
#             h.get("wind_speed_10m"),

#         "WindDirection_deg":
#             h.get("wind_direction_10m"),

#         "WindDirection_compass": [

#             deg_to_compass(
#                 x
#             )

#             for x in h.get(
#                 "wind_direction_10m",
#                 []
#             )
#         ],

#         "WindGust_kmh":
#             h.get("wind_gusts_10m"),

#         # ---------------------------------------------------------
#         # Rainfall
#         # ---------------------------------------------------------

#         "Rainfall_mm":
#             h.get("precipitation"),

#         "Rain_mm":
#             h.get("rain"),

#         # ---------------------------------------------------------
#         # Cloud
#         # ---------------------------------------------------------

#         "CloudCover_pct":
#             h.get("cloud_cover"),

#         # ---------------------------------------------------------
#         # Solar
#         # ---------------------------------------------------------

#         "SolarRadiation_Wm2":
#             h.get("shortwave_radiation"),

#         "SunshineDuration_s":
#             h.get("sunshine_duration"),

#         # ---------------------------------------------------------
#         # Visibility
#         # ---------------------------------------------------------

#         "Visibility_m":
#             h.get("visibility"),
#     })

#     return df


# # ─────────────────────────────────────────────────────────────────
# # SCRAPE MULTIPLE POINTS IN ONE API REQUEST
# # ─────────────────────────────────────────────────────────────────

# def scrape_point_batch(
#     district,
#     point_batch,
#     start_date,
#     end_date
# ):
#     """
#     Scrape multiple geographic points using one Open-Meteo
#     API request.

#     point_batch:

#         [
#             (point_id, latitude, longitude),
#             ...
#         ]

#     Returns:

#         List of DataFrames.
#     """

#     if not point_batch:

#         return []


#     # -------------------------------------------------------------
#     # Extract point information
#     # -------------------------------------------------------------

#     point_ids = [
#         p[0]
#         for p in point_batch
#     ]

#     latitudes = [
#         p[1]
#         for p in point_batch
#     ]

#     longitudes = [
#         p[2]
#         for p in point_batch
#     ]


#     # -------------------------------------------------------------
#     # Build API parameters
#     # -------------------------------------------------------------

#     params = {

#         "latitude": ",".join(
#             str(x)
#             for x in latitudes
#         ),

#         "longitude": ",".join(
#             str(x)
#             for x in longitudes
#         ),

#         "start_date":
#             start_date,

#         "end_date":
#             end_date,

#         "hourly": ",".join(
#             HOURLY_VARS
#         ),

#         "timezone":
#             "Asia/Colombo",

#         "wind_speed_unit":
#             "kmh",

#         "temperature_unit":
#             "celsius",

#         "precipitation_unit":
#             "mm",
#     }


#     # -------------------------------------------------------------
#     # Request API
#     # -------------------------------------------------------------

#     data = fetch_with_retry(
#         params
#     )


#     if data is None:

#         log.warning(
#             "No data returned for %s "
#             "(%s -> %s).",
#             district,
#             start_date,
#             end_date
#         )

#         return []


#     # -------------------------------------------------------------
#     # Multiple coordinates
#     # -------------------------------------------------------------

#     if isinstance(
#         data,
#         dict
#     ):

#         data = [
#             data
#         ]


#     results = []


#     # -------------------------------------------------------------
#     # Process each location
#     # -------------------------------------------------------------

#     for i, location_data in enumerate(
#         data
#     ):

#         if i >= len(
#             point_batch
#         ):

#             break


#         point_id = point_ids[i]

#         lat = latitudes[i]

#         lon = longitudes[i]


#         try:

#             df = build_point_dataframe(

#                 district=district,

#                 point_id=point_id,

#                 lat=lat,

#                 lon=lon,

#                 data=location_data
#             )


#             if df is not None:

#                 results.append(
#                     df
#                 )


#         except Exception as e:

#             log.warning(
#                 "Failed to process %s "
#                 "Point %s: %s",
#                 district,
#                 point_id,
#                 e
#             )


#     return results




# # ─────────────────────────────────────────────────────────────────
# # MAIN
# # ─────────────────────────────────────────────────────────────────

# def main():

#     os.makedirs(
#         OUTPUT_DIR,
#         exist_ok=True
#     )


#     # -------------------------------------------------------------
#     # Generate district grids
#     # -------------------------------------------------------------

#     district_grids = (
#         load_district_point_grids()
#     )


#     # -------------------------------------------------------------
#     # Process every district
#     # -------------------------------------------------------------

#     for district, points in (
#         district_grids.items()
#     ):


#         out_path = os.path.join(

#             OUTPUT_DIR,

#             f"{district.replace(' ', '_')}.csv"
#         )


#         # ---------------------------------------------------------
#         # Skip completed district
#         # ---------------------------------------------------------

#         if os.path.exists(
#             out_path
#         ):

#             log.info(
#                 "Skipping %s — "
#                 "output already exists at %s",
#                 district,
#                 out_path
#             )

#             continue


#         log.info(
#             "=================================================="
#         )

#         log.info(
#             "District: %s",
#             district
#         )

#         log.info(
#             "Sample points: %d",
#             len(points)
#         )

#         log.info(
#             "=================================================="
#         )


#         district_frames = []


#         # ---------------------------------------------------------
#         # Create point records
#         # ---------------------------------------------------------

#         point_records = [

#             (
#                 point_id,
#                 lat,
#                 lon
#             )

#             for point_id, (
#                 lat,
#                 lon
#             )

#             in enumerate(
#                 points,
#                 start=1
#             )
#         ]


#         # ---------------------------------------------------------
#         # Divide points into batches
#         # ---------------------------------------------------------

#         batches = [

#             point_records[i:i + BATCH_SIZE]

#             for i in range(
#                 0,
#                 len(point_records),
#                 BATCH_SIZE
#             )
#         ]


#         log.info(
#             "%s: %d points divided into %d batches.",
#             district,
#             len(point_records),
#             len(batches)
#         )


#         # ---------------------------------------------------------
#         # Process batches
#         # ---------------------------------------------------------

#         for batch_number, point_batch in enumerate(

#             batches,

#             start=1
#         ):


#             log.info(
#                 "Batch %d/%d | "
#                 "Points %d-%d",
#                 batch_number,
#                 len(batches),
#                 point_batch[0][0],
#                 point_batch[-1][0]
#             )


#             # -----------------------------------------------------
#             # Process year chunks
#             # -----------------------------------------------------

#             for start_date, end_date in (
#                 year_chunks(
#                     START_YEAR,
#                     END_YEAR
#                 )
#             ):


#                 log.info(
#                     "   Date range: %s -> %s",
#                     start_date,
#                     end_date
#                 )


#                 frames = scrape_point_batch(

#                     district=district,

#                     point_batch=point_batch,

#                     start_date=start_date,

#                     end_date=end_date
#                 )


#                 # -------------------------------------------------
#                 # Store results
#                 # -------------------------------------------------

#                 if frames:

#                     district_frames.extend(
#                         frames
#                     )

#                     log.info(
#                         "   Received %d point datasets.",
#                         len(frames)
#                     )

#                 else:

#                     log.warning(
#                         "   No data returned "
#                         "for this batch."
#                     )


#         # ---------------------------------------------------------
#         # Save district
#         # ---------------------------------------------------------

#         if district_frames:


#             full_df = pd.concat(

#                 district_frames,

#                 ignore_index=True
#             )


#             # -----------------------------------------------------
#             # Remove duplicate records
#             #
#             # This is important when restarting or when date
#             # boundaries overlap.
#             # -----------------------------------------------------

#             full_df = full_df.drop_duplicates(

#                 subset=[
#                     "PointID",
#                     "DateTime"
#                 ]
#             )


#             # -----------------------------------------------------
#             # Sort
#             # -----------------------------------------------------

#             full_df = full_df.sort_values(

#                 [
#                     "PointID",
#                     "DateTime"
#                 ]
#             ).reset_index(
#                 drop=True
#             )


#             # -----------------------------------------------------
#             # Save CSV
#             # -----------------------------------------------------

#             full_df.to_csv(

#                 out_path,

#                 index=False
#             )


#             log.info(
#                 "Saved %s | %d rows | %d points",
#                 out_path,
#                 len(full_df),
#                 full_df["PointID"].nunique()
#             )


#         else:

#             log.warning(
#                 "No data collected for %s",
#                 district
#             )


#     log.info(
#         "=================================================="
#     )

#     log.info(
#         "ALL DISTRICTS PROCESSED"
#     )

#     log.info(
#         "Output directory: %s",
#         OUTPUT_DIR
#     )

#     log.info(
#         "=================================================="
#     )


# # ─────────────────────────────────────────────────────────────────
# # RUN
# # ─────────────────────────────────────────────────────────────────

# if __name__ == "__main__":

#     main()


KeyboardInterrupt: 

In [ ]:
# # ─────────────────────────────────────────────────────────────────
# # MAIN
# # ─────────────────────────────────────────────────────────────────

# def main():

#     os.makedirs(
#         OUTPUT_DIR,
#         exist_ok=True
#     )


#     # -------------------------------------------------------------
#     # Generate district grids
#     # -------------------------------------------------------------

#     district_grids = (
#         load_district_point_grids()
#     )


#     # -------------------------------------------------------------
#     # Process every district
#     # -------------------------------------------------------------

#     for district, points in (
#         district_grids.items()
#     ):


#         out_path = os.path.join(

#             OUTPUT_DIR,

#             f"{district.replace(' ', '_')}.csv"
#         )


#         # ---------------------------------------------------------
#         # Skip completed district
#         # ---------------------------------------------------------

#         if os.path.exists(
#             out_path
#         ):

#             log.info(
#                 "Skipping %s — "
#                 "output already exists at %s",
#                 district,
#                 out_path
#             )

#             continue


#         log.info(
#             "=================================================="
#         )

#         log.info(
#             "District: %s",
#             district
#         )

#         log.info(
#             "Sample points: %d",
#             len(points)
#         )

#         log.info(
#             "=================================================="
#         )


#         district_frames = []


#         # ---------------------------------------------------------
#         # Create point records
#         # ---------------------------------------------------------

#         point_records = [

#             (
#                 point_id,
#                 lat,
#                 lon
#             )

#             for point_id, (
#                 lat,
#                 lon
#             )

#             in enumerate(
#                 points,
#                 start=1
#             )
#         ]


#         # ---------------------------------------------------------
#         # Divide points into batches
#         # ---------------------------------------------------------

#         batches = [

#             point_records[i:i + BATCH_SIZE]

#             for i in range(
#                 0,
#                 len(point_records),
#                 BATCH_SIZE
#             )
#         ]


#         log.info(
#             "%s: %d points divided into %d batches.",
#             district,
#             len(point_records),
#             len(batches)
#         )


#         # ---------------------------------------------------------
#         # Process batches
#         # ---------------------------------------------------------

#         for batch_number, point_batch in enumerate(

#             batches,

#             start=1
#         ):


#             log.info(
#                 "Batch %d/%d | "
#                 "Points %d-%d",
#                 batch_number,
#                 len(batches),
#                 point_batch[0][0],
#                 point_batch[-1][0]
#             )


#             # -----------------------------------------------------
#             # Process year chunks
#             # -----------------------------------------------------

#             for start_date, end_date in (
#                 year_chunks(
#                     START_YEAR,
#                     END_YEAR
#                 )
#             ):


#                 log.info(
#                     "   Date range: %s -> %s",
#                     start_date,
#                     end_date
#                 )


#                 frames = scrape_point_batch(

#                     district=district,

#                     point_batch=point_batch,

#                     start_date=start_date,

#                     end_date=end_date
#                 )


#                 # -------------------------------------------------
#                 # Store results
#                 # -------------------------------------------------

#                 if frames:

#                     district_frames.extend(
#                         frames
#                     )

#                     log.info(
#                         "   Received %d point datasets.",
#                         len(frames)
#                     )

#                 else:

#                     log.warning(
#                         "   No data returned "
#                         "for this batch."
#                     )


#         # ---------------------------------------------------------
#         # Save district
#         # ---------------------------------------------------------

#         if district_frames:


#             full_df = pd.concat(

#                 district_frames,

#                 ignore_index=True
#             )


#             # -----------------------------------------------------
#             # Remove duplicate records
#             #
#             # This is important when restarting or when date
#             # boundaries overlap.
#             # -----------------------------------------------------

#             full_df = full_df.drop_duplicates(

#                 subset=[
#                     "PointID",
#                     "DateTime"
#                 ]
#             )


#             # -----------------------------------------------------
#             # Sort
#             # -----------------------------------------------------

#             full_df = full_df.sort_values(

#                 [
#                     "PointID",
#                     "DateTime"
#                 ]
#             ).reset_index(
#                 drop=True
#             )


#             # -----------------------------------------------------
#             # Save CSV
#             # -----------------------------------------------------

#             full_df.to_csv(

#                 out_path,

#                 index=False
#             )


#             log.info(
#                 "Saved %s | %d rows | %d points",
#                 out_path,
#                 len(full_df),
#                 full_df["PointID"].nunique()
#             )


#         else:

#             log.warning(
#                 "No data collected for %s",
#                 district
#             )


#     log.info(
#         "=================================================="
#     )

#     log.info(
#         "ALL DISTRICTS PROCESSED"
#     )

#     log.info(
#         "Output directory: %s",
#         OUTPUT_DIR
#     )

#     log.info(
#         "=================================================="
#     )


# # ─────────────────────────────────────────────────────────────────
# # RUN
# # ─────────────────────────────────────────────────────────────────

# if __name__ == "__main__":

#     main()

AttributeError: 'NoneType' object has no attribute 'items'

In [ ]:
# ================================================================
# CELL 4 — ROBUST HOURLY WEATHER SCRAPER
# ================================================================

import os
import time
import random
import logging
import requests
import pandas as pd
from datetime import datetime


# ================================================================
# LOGGING
# ================================================================

log = logging.getLogger(__name__)

if not log.handlers:
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s"
    )


# ================================================================
# SCRAPER SETTINGS
# ================================================================

# Number of locations in one API request.
#
# We use 10 because Open-Meteo supports multiple coordinates
# in one request. Using 2 would create too many API calls.
BATCH_SIZE = 10


# Delay between successful requests.
#
# This is intentionally conservative to reduce 429 errors.
REQUEST_DELAY = 15.0


# Number of attempts for temporary failures.
MAX_RETRIES = 6


# Initial retry delay after 429/server errors.
BASE_SLEEP = 60.0


# Maximum retry waiting time.
MAX_SLEEP = 600.0


# HTTP timeout.
REQUEST_TIMEOUT = 180


# ================================================================
# CHECKPOINT DIRECTORY
# ================================================================

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

CHECKPOINT_DIR = os.path.join(
    OUTPUT_DIR,
    "_checkpoints"
)

os.makedirs(
    CHECKPOINT_DIR,
    exist_ok=True
)


# ================================================================
# FAILED REQUEST LOG
# ================================================================

FAILED_LOG_PATH = os.path.join(
    CHECKPOINT_DIR,
    "failed_requests.csv"
)


# ================================================================
# HOURLY VARIABLES
# ================================================================

HOURLY_VARS = [

    "temperature_2m",

    "apparent_temperature",

    "relative_humidity_2m",

    "dew_point_2m",

    "pressure_msl",

    "wind_speed_10m",

    "wind_direction_10m",

    "wind_gusts_10m",

    "precipitation",

    "rain",

    "cloud_cover",

    "shortwave_radiation",

    "sunshine_duration",

    "visibility",
]


# ================================================================
# SAFE FILE NAME
# ================================================================

def safe_filename(name):

    return (
        str(name)
        .replace(" ", "_")
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
    )


# ================================================================
# CHECKPOINT PATH
# ================================================================

def get_checkpoint_path(district):

    return os.path.join(
        CHECKPOINT_DIR,
        f"{safe_filename(district)}_progress.csv"
    )


# ================================================================
# LOAD COMPLETED CHUNKS
# ================================================================

def load_completed_chunks(district):

    path = get_checkpoint_path(
        district
    )

    if not os.path.exists(path):

        return set()

    try:

        df = pd.read_csv(path)

        completed = set()

        for _, row in df.iterrows():

            completed.add(
                (
                    int(row["Batch"]),
                    str(row["StartDate"]),
                    str(row["EndDate"])
                )
            )

        return completed

    except Exception as e:

        log.warning(
            "Could not read checkpoint for %s: %s",
            district,
            e
        )

        return set()


# ================================================================
# MARK CHUNK COMPLETED
# ================================================================

def mark_chunk_completed(
    district,
    batch_number,
    start_date,
    end_date
):

    path = get_checkpoint_path(
        district
    )

    row = pd.DataFrame({

        "Batch": [
            batch_number
        ],

        "StartDate": [
            start_date
        ],

        "EndDate": [
            end_date
        ],

        "CompletedAt": [
            datetime.now().isoformat()
        ]
    })

    if os.path.exists(path):

        row.to_csv(
            path,
            mode="a",
            header=False,
            index=False
        )

    else:

        row.to_csv(
            path,
            index=False
        )


# ================================================================
# RECORD FAILED REQUEST
# ================================================================

def record_failed_request(
    district,
    batch_number,
    start_date,
    end_date,
    error
):

    row = pd.DataFrame({

        "District": [
            district
        ],

        "Batch": [
            batch_number
        ],

        "StartDate": [
            start_date
        ],

        "EndDate": [
            end_date
        ],

        "Error": [
            str(error)
        ],

        "RecordedAt": [
            datetime.now().isoformat()
        ]
    })

    if os.path.exists(
        FAILED_LOG_PATH
    ):

        row.to_csv(
            FAILED_LOG_PATH,
            mode="a",
            header=False,
            index=False
        )

    else:

        row.to_csv(
            FAILED_LOG_PATH,
            index=False
        )


# ================================================================
# API REQUEST WITH RETRY
# ================================================================

def fetch_with_retry(
    params,
    district=None,
    batch_number=None,
    start_date=None,
    end_date=None
):

    """
    Robust Open-Meteo request.

    Handles:
        429 rate limiting
        500
        502
        503
        504
        timeout
        connection errors
    """

    for attempt in range(
        1,
        MAX_RETRIES + 1
    ):

        try:

            print(
                f"🌐 API request | "
                f"{district or ''} | "
                f"Batch {batch_number or ''} | "
                f"{start_date or ''} → {end_date or ''} | "
                f"Attempt {attempt}/{MAX_RETRIES}",
                flush=True
            )

            response = requests.get(
                BASE_URL,
                params=params,
                timeout=REQUEST_TIMEOUT
            )


            # ====================================================
            # 429 RATE LIMIT
            # ====================================================

            if response.status_code == 429:

                retry_after = response.headers.get(
                    "Retry-After"
                )


                if retry_after:

                    try:

                        sleep_for = float(
                            retry_after
                        )

                    except ValueError:

                        sleep_for = (
                            BASE_SLEEP *
                            (2 ** (attempt - 1))
                        )

                else:

                    sleep_for = (
                        BASE_SLEEP *
                        (2 ** (attempt - 1))
                    )


                sleep_for = min(
                    sleep_for,
                    MAX_SLEEP
                )


                sleep_for += random.uniform(
                    5,
                    15
                )


                print(
                    f"⚠️ Rate limited (429). "
                    f"Attempt {attempt}/{MAX_RETRIES}. "
                    f"Waiting {sleep_for:.1f} seconds...",
                    flush=True
                )


                time.sleep(
                    sleep_for
                )

                continue


            # ====================================================
            # TEMPORARY SERVER ERRORS
            # ====================================================

            if response.status_code in (
                500,
                502,
                503,
                504
            ):

                sleep_for = (
                    BASE_SLEEP *
                    (2 ** (attempt - 1))
                    +
                    random.uniform(5, 15)
                )

                sleep_for = min(
                    sleep_for,
                    MAX_SLEEP
                )

                print(
                    f"⚠️ Server error "
                    f"{response.status_code}. "
                    f"Waiting {sleep_for:.1f} seconds...",
                    flush=True
                )

                time.sleep(
                    sleep_for
                )

                continue


            # ====================================================
            # OTHER HTTP ERRORS
            # ====================================================

            response.raise_for_status()


            # ====================================================
            # JSON RESPONSE
            # ====================================================

            data = response.json()


            # ====================================================
            # SUCCESS DELAY
            # ====================================================

            delay = (
                REQUEST_DELAY
                +
                random.uniform(1, 5)
            )

            print(
                f"✅ Request successful. "
                f"Next request in {delay:.1f}s",
                flush=True
            )

            time.sleep(
                delay
            )


            return data


        # ========================================================
        # TIMEOUT
        # ========================================================

        except requests.exceptions.Timeout:

            sleep_for = (
                BASE_SLEEP *
                (2 ** (attempt - 1))
                +
                random.uniform(5, 15)
            )

            sleep_for = min(
                sleep_for,
                MAX_SLEEP
            )

            print(
                f"⚠️ Timeout. "
                f"Retrying in {sleep_for:.1f}s...",
                flush=True
            )

            time.sleep(
                sleep_for
            )


        # ========================================================
        # CONNECTION ERROR
        # ========================================================

        except requests.exceptions.ConnectionError:

            sleep_for = (
                BASE_SLEEP *
                (2 ** (attempt - 1))
                +
                random.uniform(5, 15)
            )

            sleep_for = min(
                sleep_for,
                MAX_SLEEP
            )

            print(
                f"⚠️ Connection error. "
                f"Retrying in {sleep_for:.1f}s...",
                flush=True
            )

            time.sleep(
                sleep_for
            )


        # ========================================================
        # REQUEST ERROR
        # ========================================================

        except requests.exceptions.RequestException as e:

            print(
                f"❌ Request failed: {e}",
                flush=True
            )

            break


        # ========================================================
        # UNEXPECTED ERROR
        # ========================================================

        except Exception as e:

            print(
                f"❌ Unexpected error: {e}",
                flush=True
            )

            break


    print(
        f"❌ FAILED after {MAX_RETRIES} attempts.",
        flush=True
    )

    return None


# ================================================================
# BUILD DATAFRAME
# ================================================================

def build_point_dataframe(
    district,
    point_id,
    lat,
    lon,
    data
):

    if data is None:

        return None


    if "hourly" not in data:

        log.warning(
            "%s Point %s: No hourly data.",
            district,
            point_id
        )

        return None


    h = data["hourly"]


    # ------------------------------------------------------------
    # TIME
    # ------------------------------------------------------------

    times = pd.to_datetime(
        h["time"]
    )


    # ------------------------------------------------------------
    # DATAFRAME
    # ------------------------------------------------------------

    df = pd.DataFrame({

        "District":
            district,

        "PointID":
            point_id,

        "Latitude":
            lat,

        "Longitude":
            lon,

        "Elevation_m":
            data.get("elevation"),

        "DateTime":
            h["time"],

        "Date":
            times.date,

        "Day":
            times.day,

        "Month":
            times.month,

        "Year":
            times.year,

        "Time":
            times.strftime("%H:%M"),

        "Season": [
            get_season(m)
            for m in times.month
        ],

        # --------------------------------------------------------
        # TEMPERATURE
        # --------------------------------------------------------

        "Temperature_C":
            h.get("temperature_2m"),

        "FeelsLike_C":
            h.get("apparent_temperature"),

        # --------------------------------------------------------
        # HUMIDITY
        # --------------------------------------------------------

        "Humidity_pct":
            h.get("relative_humidity_2m"),

        "DewPoint_C":
            h.get("dew_point_2m"),

        # --------------------------------------------------------
        # PRESSURE
        # --------------------------------------------------------

        "Pressure_hPa":
            h.get("pressure_msl"),

        # --------------------------------------------------------
        # WIND
        # --------------------------------------------------------

        "WindSpeed_kmh":
            h.get("wind_speed_10m"),

        "WindDirection_deg":
            h.get("wind_direction_10m"),

        "WindDirection_compass": [
            deg_to_compass(x)
            for x in h.get(
                "wind_direction_10m",
                []
            )
        ],

        "WindGust_kmh":
            h.get("wind_gusts_10m"),

        # --------------------------------------------------------
        # RAIN
        # --------------------------------------------------------

        "Rainfall_mm":
            h.get("precipitation"),

        "Rain_mm":
            h.get("rain"),

        # --------------------------------------------------------
        # CLOUD
        # --------------------------------------------------------

        "CloudCover_pct":
            h.get("cloud_cover"),

        # --------------------------------------------------------
        # SOLAR
        # --------------------------------------------------------

        "SolarRadiation_Wm2":
            h.get("shortwave_radiation"),

        "SunshineDuration_s":
            h.get("sunshine_duration"),

        # --------------------------------------------------------
        # VISIBILITY
        # --------------------------------------------------------

        "Visibility_m":
            h.get("visibility"),
    })


    return df


# ================================================================
# BUILD API PARAMETERS
# ================================================================

def build_api_params(
    point_batch,
    start_date,
    end_date
):

    latitudes = [
        p[1]
        for p in point_batch
    ]

    longitudes = [
        p[2]
        for p in point_batch
    ]


    return {

        "latitude": ",".join(
            str(x)
            for x in latitudes
        ),

        "longitude": ",".join(
            str(x)
            for x in longitudes
        ),

        "start_date":
            start_date,

        "end_date":
            end_date,

        "hourly": ",".join(
            HOURLY_VARS
        ),

        "timezone":
            "Asia/Colombo",

        "wind_speed_unit":
            "kmh",

        "temperature_unit":
            "celsius",

        "precipitation_unit":
            "mm",
    }


# ================================================================
# SCRAPE POINT BATCH
# ================================================================

def scrape_point_batch(
    district,
    point_batch,
    start_date,
    end_date,
    batch_number
):

    if not point_batch:

        return []


    point_ids = [
        p[0]
        for p in point_batch
    ]

    latitudes = [
        p[1]
        for p in point_batch
    ]

    longitudes = [
        p[2]
        for p in point_batch
    ]


    params = build_api_params(
        point_batch,
        start_date,
        end_date
    )


    data = fetch_with_retry(

        params,

        district=district,

        batch_number=batch_number,

        start_date=start_date,

        end_date=end_date
    )


    if data is None:

        return None


    # ------------------------------------------------------------
    # SINGLE LOCATION
    # ------------------------------------------------------------

    if isinstance(data, dict):

        data = [data]


    results = []


    # ------------------------------------------------------------
    # PROCESS LOCATIONS
    # ------------------------------------------------------------

    for i, location_data in enumerate(data):

        if i >= len(point_batch):

            break


        try:

            df = build_point_dataframe(

                district=district,

                point_id=point_ids[i],

                lat=latitudes[i],

                lon=longitudes[i],

                data=location_data

            )


            if df is not None:

                results.append(df)


        except Exception as e:

            log.warning(
                "Processing error | "
                "%s | Point %s | %s",
                district,
                point_ids[i],
                e
            )


    return results


# ================================================================
# APPEND DATA TO CSV
# ================================================================

def append_to_district_csv(
    district,
    frames
):

    if not frames:

        return


    out_path = os.path.join(

        OUTPUT_DIR,

        f"{safe_filename(district)}.csv"
    )


    new_df = pd.concat(
        frames,
        ignore_index=True
    )


    # ------------------------------------------------------------
    # REMOVE DUPLICATES IN NEW DATA
    # ------------------------------------------------------------

    new_df = new_df.drop_duplicates(

        subset=[
            "PointID",
            "DateTime"
        ]
    )


    # ------------------------------------------------------------
    # EXISTING FILE
    # ------------------------------------------------------------

    if os.path.exists(out_path):

        try:

            existing_df = pd.read_csv(
                out_path
            )


            combined = pd.concat(

                [
                    existing_df,
                    new_df
                ],

                ignore_index=True
            )


            combined = combined.drop_duplicates(

                subset=[
                    "PointID",
                    "DateTime"
                ]
            )


            combined = combined.sort_values(

                [
                    "PointID",
                    "DateTime"
                ]

            ).reset_index(
                drop=True
            )


            combined.to_csv(

                out_path,

                index=False
            )


        except Exception as e:

            log.error(
                "Could not update %s: %s",
                out_path,
                e
            )

            return


    # ------------------------------------------------------------
    # NEW FILE
    # ------------------------------------------------------------

    else:

        new_df = new_df.sort_values(

            [
                "PointID",
                "DateTime"
            ]

        ).reset_index(
            drop=True
        )


        new_df.to_csv(

            out_path,

            index=False
        )


    print(
        f"💾 Saved {len(new_df):,} rows → {out_path}",
        flush=True
    )


# ================================================================
# CREATE BATCHES
# ================================================================

def create_batches(
    point_records,
    batch_size
):

    return [

        point_records[i:i + batch_size]

        for i in range(
            0,
            len(point_records),
            batch_size
        )
    ]


# ================================================================
# MAIN SCRAPER
# ================================================================

def main():

    os.makedirs(
        OUTPUT_DIR,
        exist_ok=True
    )


    print(
        "\n🚀 STARTING HOURLY WEATHER SCRAPER\n",
        flush=True
    )


    # ============================================================
    # LOAD DISTRICT POINTS
    # ============================================================

    district_grids = (
        load_district_point_grids()
    )


    print(
        f"📍 Districts loaded: "
        f"{len(district_grids)}",
        flush=True
    )


    # ============================================================
    # DISTRICTS
    # ============================================================

    for district, points in district_grids.items():

        print(
            "\n" + "=" * 70,
            flush=True
        )

        print(
            f"📍 DISTRICT: {district}",
            flush=True
        )

        print(
            "=" * 70,
            flush=True
        )


        # --------------------------------------------------------
        # POINT RECORDS
        # --------------------------------------------------------

        point_records = [

            (
                point_id,
                lat,
                lon
            )

            for point_id, (lat, lon)

            in enumerate(
                points,
                start=1
            )
        ]


        print(
            f"📌 Points: {len(point_records)}",
            flush=True
        )


        # --------------------------------------------------------
        # BATCHES
        # --------------------------------------------------------

        batches = create_batches(

            point_records,

            BATCH_SIZE

        )


        print(
            f"📦 Batches: {len(batches)}",
            flush=True
        )


        # --------------------------------------------------------
        # CHECKPOINT
        # --------------------------------------------------------

        completed_chunks = (
            load_completed_chunks(
                district
            )
        )


        print(
            f"✅ Completed chunks: "
            f"{len(completed_chunks)}",
            flush=True
        )


        # ========================================================
        # PROCESS BATCHES
        # ========================================================

        for batch_number, point_batch in enumerate(

            batches,

            start=1

        ):

            print(
                "\n" +
                "-" * 70,
                flush=True
            )

            print(
                f"📦 Batch "
                f"{batch_number}/{len(batches)} | "
                f"Points "
                f"{point_batch[0][0]}-"
                f"{point_batch[-1][0]}",
                flush=True
            )

            print(
                "-" * 70,
                flush=True
            )


            # ====================================================
            # YEAR CHUNKS
            # ====================================================

            for start_date, end_date in year_chunks(

                START_YEAR,

                END_YEAR

            ):


                chunk_key = (

                    batch_number,

                    str(start_date),

                    str(end_date)

                )


                # ------------------------------------------------
                # SKIP COMPLETED
                # ------------------------------------------------

                if chunk_key in completed_chunks:

                    print(
                        f"⏭️ SKIP | "
                        f"Batch {batch_number} | "
                        f"{start_date} → {end_date}",
                        flush=True
                    )

                    continue


                print(
                    f"\n⬇️ DOWNLOAD | "
                    f"Batch {batch_number} | "
                    f"{start_date} → {end_date}",
                    flush=True
                )


                # ------------------------------------------------
                # REQUEST
                # ------------------------------------------------

                frames = scrape_point_batch(

                    district=district,

                    point_batch=point_batch,

                    start_date=start_date,

                    end_date=end_date,

                    batch_number=batch_number

                )


                # ------------------------------------------------
                # FAILURE
                # ------------------------------------------------

                if frames is None:

                    print(
                        f"❌ FAILED | "
                        f"Batch {batch_number} | "
                        f"{start_date} → {end_date}",
                        flush=True
                    )


                    record_failed_request(

                        district,

                        batch_number,

                        start_date,

                        end_date,

                        "All API retries failed."

                    )

                    continue


                # ------------------------------------------------
                # NO DATA
                # ------------------------------------------------

                if not frames:

                    print(
                        f"⚠️ No data returned.",
                        flush=True
                    )


                    record_failed_request(

                        district,

                        batch_number,

                        start_date,

                        end_date,

                        "No data returned."

                    )

                    continue


                # ------------------------------------------------
                # SAVE
                # ------------------------------------------------

                append_to_district_csv(

                    district,

                    frames

                )


                # ------------------------------------------------
                # CHECKPOINT
                # ------------------------------------------------

                mark_chunk_completed(

                    district,

                    batch_number,

                    start_date,

                    end_date

                )


                completed_chunks.add(
                    chunk_key
                )


                print(
                    f"✅ COMPLETED | "
                    f"{district} | "
                    f"Batch {batch_number} | "
                    f"{start_date} → {end_date}",
                    flush=True
                )


        # ========================================================
        # DISTRICT SUMMARY
        # ========================================================

        out_path = os.path.join(

            OUTPUT_DIR,

            f"{safe_filename(district)}.csv"

        )


        if os.path.exists(out_path):

            try:

                df = pd.read_csv(
                    out_path
                )


                print(
                    "\n" +
                    "=" * 70,
                    flush=True
                )

                print(
                    f"✅ DISTRICT FINISHED: {district}",
                    flush=True
                )

                print(
                    f"Rows: {len(df):,}",
                    flush=True
                )

                print(
                    f"Points: "
                    f"{df['PointID'].nunique():,}",
                    flush=True
                )

                print(
                    f"Date range: "
                    f"{df['DateTime'].min()} → "
                    f"{df['DateTime'].max()}",
                    flush=True
                )

                print(
                    f"File: {out_path}",
                    flush=True
                )

                print(
                    "=" * 70,
                    flush=True
                )


            except Exception as e:

                log.warning(
                    "Could not summarize %s: %s",
                    district,
                    e
                )


    # ============================================================
    # FINISHED
    # ============================================================

    print(
        "\n" +
        "=" * 70,
        flush=True
    )

    print(
        "🎉 ALL DISTRICTS PROCESSED",
        flush=True
    )

    print(
        f"📁 Output: {OUTPUT_DIR}",
        flush=True
    )

    print(
        f"📋 Checkpoints: {CHECKPOINT_DIR}",
        flush=True
    )

    print(
        f"❌ Failed requests: {FAILED_LOG_PATH}",
        flush=True
    )

    print(
        "=" * 70,
        flush=True
    )

In [ ]:
test_params = {
    "latitude": "7.2906",
    "longitude": "80.6337",
    "start_date": "2020-01-01",
    "end_date": "2020-01-07",
    "hourly": ",".join(HOURLY_VARS),
    "timezone": "Asia/Colombo",
    "wind_speed_unit": "kmh",
    "temperature_unit": "celsius",
    "precipitation_unit": "mm",
}

print("Testing API connection...")

response = requests.get(
    BASE_URL,
    params=test_params,
    timeout=180
)

print("Status:", response.status_code)
print("Response:", response.text[:500])

Testing API connection...
Status: 200
Response: {"latitude":7.275923,"longitude":80.62659,"generationtime_ms":98.14369678497314,"utc_offset_seconds":19800,"timezone":"Asia/Colombo","timezone_abbreviation":"GMT+5:30","elevation":499.0,"hourly_units":{"time":"iso8601","temperature_2m":"°C","apparent_temperature":"°C","relative_humidity_2m":"%","dew_point_2m":"°C","pressure_msl":"hPa","wind_speed_10m":"km/h","wind_direction_10m":"°","wind_gusts_10m":"km/h","precipitation":"mm","rain":"mm","cloud_cover":"%","shortwave_radiation":"W/m²","sunshine_


In [ ]:
# ================================================================
# TEST MULTIPLE LOCATIONS
# ================================================================

test_params = {
    "latitude": "7.2906,7.8731,6.9271,6.0535,8.3114",
    "longitude": "80.6337,80.7718,79.8612,80.2210,80.4037",

    "start_date": "2020-01-01",
    "end_date": "2020-01-07",

    "hourly": ",".join(HOURLY_VARS),

    "timezone": "Asia/Colombo",

    "wind_speed_unit": "kmh",

    "temperature_unit": "celsius",

    "precipitation_unit": "mm",
}


print("Testing multiple-location request...")


response = requests.get(
    BASE_URL,
    params=test_params,
    timeout=180
)


print(
    "Status:",
    response.status_code
)


if response.status_code == 200:

    data = response.json()

    print(
        "Number of locations returned:",
        len(data) if isinstance(data, list) else 1
    )

    print(
        "✅ MULTIPLE-LOCATION TEST SUCCESSFUL"
    )

else:

    print(
        "❌ Request failed"
    )

    print(
        response.text[:1000]
    )

Testing multiple-location request...
Status: 200
Number of locations returned: 5
✅ MULTIPLE-LOCATION TEST SUCCESSFUL


In [ ]:
# ================================================================
# CELL 4 — HOURLY WEATHER SCRAPING
# ================================================================
#
# Robust Open-Meteo historical hourly weather scraper
#
# Features:
#   - Multiple geographic points per API request
#   - 429 rate-limit handling
#   - Retry-After support
#   - Hourly quota detection
#   - Conservative exponential backoff
#   - Checkpointing
#   - Resume after interruption
#   - Failed-request logging
#   - Incremental CSV saving
#   - Duplicate removal
#   - Clear batch/year progress
#
# IMPORTANT:
# This cell assumes these are already defined:
#
#   BASE_URL
#   OUTPUT_DIR
#   START_YEAR
#   END_YEAR
#   load_district_point_grids()
#   year_chunks()
#   get_season()
#   deg_to_compass()
#
# ================================================================


# ================================================================
# IMPORTS
# ================================================================

import os
import time
import random
import logging
import requests
import pandas as pd

from datetime import datetime


# ================================================================
# LOGGING
# ================================================================

log = logging.getLogger(__name__)

if not log.handlers:

    logging.basicConfig(

        level=logging.INFO,

        format=(
            "%(asctime)s | "
            "%(levelname)s | "
            "%(message)s"
        )
    )


# ================================================================
# SCRAPER SETTINGS
# ================================================================

# ------------------------------------------------
# LOCATIONS PER API REQUEST
# ------------------------------------------------
#
# Example:
#
# 28 points
# BATCH_SIZE = 10
#
# Batch 1 = points 1-10
# Batch 2 = points 11-20
# Batch 3 = points 21-28
#
BATCH_SIZE = 10


# ------------------------------------------------
# NORMAL DELAY AFTER SUCCESS
# ------------------------------------------------
#
# This is deliberately higher than a very aggressive
# scraper because you are making many historical requests.
#
REQUEST_DELAY = 20.0


# ------------------------------------------------
# RANDOM DELAY AFTER SUCCESS
# ------------------------------------------------

SUCCESS_JITTER_MIN = 3.0
SUCCESS_JITTER_MAX = 8.0


# ------------------------------------------------
# MAXIMUM RETRIES FOR TEMPORARY ERRORS
# ------------------------------------------------
#
# Keep this relatively low.
#
# If the API is actually rate-limiting us, repeatedly
# retrying can make the situation worse.
#
MAX_RETRIES = 3


# ------------------------------------------------
# BASE RETRY DELAY
# ------------------------------------------------

BASE_SLEEP = 120.0


# ------------------------------------------------
# MAXIMUM RETRY DELAY
# ------------------------------------------------

MAX_SLEEP = 900.0


# ------------------------------------------------
# EXTRA BUFFER AFTER RETRY-AFTER
# ------------------------------------------------
#
# If API says:
#
# Retry-After = 72 seconds
#
# we wait:
#
# 72 + random safety buffer
#
# This reduces the chance of immediately receiving
# another 429.
#
RETRY_AFTER_BUFFER_MIN = 15.0
RETRY_AFTER_BUFFER_MAX = 30.0


# ------------------------------------------------
# JITTER FOR EXPONENTIAL BACKOFF
# ------------------------------------------------

JITTER_MIN = 15.0
JITTER_MAX = 30.0


# ------------------------------------------------
# HTTP TIMEOUT
# ------------------------------------------------

REQUEST_TIMEOUT = 180


# ------------------------------------------------
# QUOTA STOP BUFFER
# ------------------------------------------------
#
# If Open-Meteo explicitly says the hourly quota has
# been exceeded, do NOT repeatedly hit the API.
#
# Instead, stop processing requests for this run.
#
# The next run can resume using checkpoints.
#
QUOTA_STOP_ON_EXHAUSTION = True


# ================================================================
# OUTPUT DIRECTORIES
# ================================================================

os.makedirs(

    OUTPUT_DIR,

    exist_ok=True

)


CHECKPOINT_DIR = os.path.join(

    OUTPUT_DIR,

    "_checkpoints"

)


os.makedirs(

    CHECKPOINT_DIR,

    exist_ok=True

)


FAILED_LOG_PATH = os.path.join(

    CHECKPOINT_DIR,

    "failed_requests.csv"

)


# ================================================================
# HOURLY VARIABLES
# ================================================================

HOURLY_VARS = [

    "temperature_2m",

    "apparent_temperature",

    "relative_humidity_2m",

    "dew_point_2m",

    "pressure_msl",

    "wind_speed_10m",

    "wind_direction_10m",

    "wind_gusts_10m",

    "precipitation",

    "rain",

    "cloud_cover",

    "shortwave_radiation",

    "sunshine_duration",

    "visibility",

]


# ================================================================
# GLOBAL QUOTA STATUS
# ================================================================
#
# This prevents the scraper from continuing to send
# requests after the API explicitly reports that the
# hourly quota has been exhausted.
#
HOURLY_QUOTA_EXHAUSTED = False


# ================================================================
# SAFE FILE NAME
# ================================================================

def safe_filename(name):

    return (

        str(name)

        .replace(" ", "_")

        .replace("/", "_")

        .replace("\\", "_")

        .replace(":", "_")

    )


# ================================================================
# CHECKPOINT PATH
# ================================================================

def get_checkpoint_path(district):

    return os.path.join(

        CHECKPOINT_DIR,

        f"{safe_filename(district)}_progress.csv"

    )


# ================================================================
# LOAD COMPLETED CHUNKS
# ================================================================

def load_completed_chunks(district):

    path = get_checkpoint_path(

        district

    )


    if not os.path.exists(path):

        return set()


    try:

        df = pd.read_csv(

            path

        )


        completed = set()


        for _, row in df.iterrows():

            completed.add(

                (

                    int(row["Batch"]),

                    str(row["StartDate"]),

                    str(row["EndDate"])

                )

            )


        return completed


    except Exception as e:

        print(

            f"Could not read checkpoint "
            f"for {district}: {e}",

            flush=True

        )

        return set()


# ================================================================
# MARK CHUNK COMPLETED
# ================================================================

def mark_chunk_completed(

    district,

    batch_number,

    start_date,

    end_date

):

    path = get_checkpoint_path(

        district

    )


    row = pd.DataFrame({

        "Batch": [

            batch_number

        ],

        "StartDate": [

            start_date

        ],

        "EndDate": [

            end_date

        ],

        "CompletedAt": [

            datetime.now().isoformat()

        ]

    })


    if os.path.exists(path):

        row.to_csv(

            path,

            mode="a",

            header=False,

            index=False

        )

    else:

        row.to_csv(

            path,

            index=False

        )


# ================================================================
# RECORD FAILED REQUEST
# ================================================================

def record_failed_request(

    district,

    batch_number,

    start_date,

    end_date,

    error

):

    row = pd.DataFrame({

        "District": [

            district

        ],

        "Batch": [

            batch_number

        ],

        "StartDate": [

            start_date

        ],

        "EndDate": [

            end_date

        ],

        "Error": [

            str(error)

        ],

        "RecordedAt": [

            datetime.now().isoformat()

        ]

    })


    if os.path.exists(

        FAILED_LOG_PATH

    ):

        row.to_csv(

            FAILED_LOG_PATH,

            mode="a",

            header=False,

            index=False

        )

    else:

        row.to_csv(

            FAILED_LOG_PATH,

            index=False

        )


# ================================================================
# GET API ERROR MESSAGE
# ================================================================

def get_api_error_message(response):

    try:

        data = response.json()

        if isinstance(data, dict):

            return (

                data.get("reason")

                or

                data.get("error")

                or

                str(data)

            )

    except Exception:

        pass


    return response.text[:500]


# ================================================================
# CHECK HOURLY QUOTA ERROR
# ================================================================

def is_hourly_quota_error(response):

    """
    Detect Open-Meteo hourly quota exhaustion.
    """

    try:

        data = response.json()

        if isinstance(data, dict):

            reason = str(

                data.get(

                    "reason",

                    ""

                )

            ).lower()


            if (

                "hourly api request limit" in reason

                or

                "request limit exceeded" in reason

                or

                "try again in the next hour" in reason

                or

                "hourly quota" in reason

                or

                "quota exceeded" in reason

            ):

                return True


    except Exception:

        pass


    text = response.text.lower()


    return (

        "hourly api request limit" in text

        or

        "try again in the next hour" in text

        or

        "hourly quota" in text

        or

        "quota exceeded" in text

    )


# ================================================================
# CALCULATE EXPONENTIAL BACKOFF
# ================================================================

def calculate_retry_delay(attempt):

    """
    Conservative exponential backoff.

    Attempt 1:
        120 sec + jitter

    Attempt 2:
        240 sec + jitter

    Attempt 3:
        480 sec + jitter

    Maximum:
        900 sec
    """

    sleep_for = (

        BASE_SLEEP *

        (2 ** (attempt - 1))

    )


    sleep_for = min(

        sleep_for,

        MAX_SLEEP

    )


    sleep_for += random.uniform(

        JITTER_MIN,

        JITTER_MAX

    )


    return sleep_for


# ================================================================
# API REQUEST WITH ROBUST RETRY
# ================================================================

def fetch_with_retry(

    params,

    district=None,

    batch_number=None,

    start_date=None,

    end_date=None

):

    global HOURLY_QUOTA_EXHAUSTED


    # ------------------------------------------------------------
    # DO NOT SEND REQUEST IF GLOBAL QUOTA IS ALREADY EXHAUSTED
    # ------------------------------------------------------------

    if HOURLY_QUOTA_EXHAUSTED:

        print(

            "\n API HOURLY QUOTA ALREADY EXHAUSTED."

            "\n Skipping further API requests "
            "for this run.",

            flush=True

        )

        return None


    # ============================================================
    # RETRY LOOP
    # ============================================================

    for attempt in range(

        1,

        MAX_RETRIES + 1

    ):

        try:

            print(

                f"\n API REQUEST | "
                f"{district} | "
                f"Batch {batch_number} | "
                f"{start_date} → {end_date} | "
                f"Attempt {attempt}/{MAX_RETRIES}",

                flush=True

            )


            response = requests.get(

                BASE_URL,

                params=params,

                timeout=REQUEST_TIMEOUT

            )


            # ====================================================
            # SUCCESS
            # ====================================================

            if response.status_code == 200:

                try:

                    data = response.json()

                except ValueError as e:

                    print(

                        f"Invalid JSON response: {e}",

                        flush=True

                    )

                    return None


                delay = (

                    REQUEST_DELAY

                    +

                    random.uniform(

                        SUCCESS_JITTER_MIN,

                        SUCCESS_JITTER_MAX

                    )

                )


                print(

                    f"SUCCESS | "
                    f"Waiting {delay:.1f} seconds "
                    f"before next request...",

                    flush=True

                )


                time.sleep(

                    delay

                )


                return data


            # ====================================================
            # 429 RATE LIMIT
            # ====================================================

            if response.status_code == 429:

                reason = get_api_error_message(

                    response

                )


                print(

                    "\n RATE LIMITED (429)",

                    flush=True

                )


                print(

                    f"District : {district}",

                    flush=True

                )


                print(

                    f"Batch    : {batch_number}",

                    flush=True

                )


                print(

                    f"Period   : "
                    f"{start_date} → {end_date}",

                    flush=True

                )


                print(

                    f"Attempt  : "
                    f"{attempt}/{MAX_RETRIES}",

                    flush=True

                )


                print(

                    f"API reason: {reason}",

                    flush=True

                )


                # =================================================
                # HOURLY QUOTA EXHAUSTED
                # =================================================

                if is_hourly_quota_error(

                    response

                ):

                    HOURLY_QUOTA_EXHAUSTED = True


                    print(

                        "\n"
                        + "=" * 70,

                        flush=True

                    )


                    print(

                        "HOURLY API QUOTA EXHAUSTED",

                        flush=True

                    )


                    print(

                        "The API has explicitly reported that "
                        "the hourly request quota is exhausted.",

                        flush=True

                    )


                    print(

                        "No additional retry attempts "
                        "will be made.",

                        flush=True

                    )


                    print(

                        "The current chunk will NOT be "
                        "marked as completed.",

                        flush=True

                    )


                    print(

                        "Restart the scraper after the "
                        "API quota resets.",

                        flush=True

                    )


                    print(

                        "=" * 70,

                        flush=True

                    )


                    return None


                # =================================================
                # TEMPORARY 429
                # =================================================

                retry_after = response.headers.get(

                    "Retry-After"

                )


                sleep_for = None


                # -------------------------------------------------
                # API PROVIDED RETRY-AFTER
                # -------------------------------------------------

                if retry_after:

                    try:

                        api_wait = float(

                            retry_after

                        )


                        safety_buffer = random.uniform(

                            RETRY_AFTER_BUFFER_MIN,

                            RETRY_AFTER_BUFFER_MAX

                        )


                        sleep_for = (

                            api_wait

                            +

                            safety_buffer

                        )


                        sleep_for = min(

                            sleep_for,

                            MAX_SLEEP

                        )


                        print(

                            f"Retry-After received: "
                            f"{api_wait:.1f} seconds",

                            flush=True

                        )


                        print(

                            f"Safety buffer: "
                            f"{safety_buffer:.1f} seconds",

                            flush=True

                        )


                    except (

                        ValueError,

                        TypeError

                    ):

                        sleep_for = None


                # -------------------------------------------------
                # NO VALID RETRY-AFTER
                # -------------------------------------------------

                if sleep_for is None:

                    sleep_for = calculate_retry_delay(

                        attempt

                    )


                # -------------------------------------------------
                # WAIT
                # -------------------------------------------------

                print(

                    f"Waiting "
                    f"{sleep_for / 60:.1f} minutes "
                    f"before retry...",

                    flush=True

                )


                time.sleep(

                    sleep_for

                )


                continue


            # ====================================================
            # SERVER ERRORS
            # ====================================================

            if response.status_code in (

                500,

                502,

                503,

                504

            ):

                sleep_for = calculate_retry_delay(

                    attempt

                )


                print(

                    f"SERVER ERROR "
                    f"{response.status_code} | "

                    f"Waiting "
                    f"{sleep_for / 60:.1f} minutes...",

                    flush=True

                )


                time.sleep(

                    sleep_for

                )


                continue


            # ====================================================
            # OTHER HTTP ERRORS
            # ====================================================

            reason = get_api_error_message(

                response

            )


            print(

                f"HTTP ERROR "
                f"{response.status_code} | "
                f"{reason}",

                flush=True

            )


            return None


        # ========================================================
        # TIMEOUT
        # ========================================================

        except requests.exceptions.Timeout:

            sleep_for = calculate_retry_delay(

                attempt

            )


            print(

                f"TIMEOUT | "
                f"Retrying in "
                f"{sleep_for / 60:.1f} minutes...",

                flush=True

            )


            time.sleep(

                sleep_for

            )


        # ========================================================
        # CONNECTION ERROR
        # ========================================================

        except requests.exceptions.ConnectionError as e:

            sleep_for = calculate_retry_delay(

                attempt

            )


            print(

                f"CONNECTION ERROR | "
                f"{e} | "
                f"Retrying in "
                f"{sleep_for / 60:.1f} minutes...",

                flush=True

            )


            time.sleep(

                sleep_for

            )


        # ========================================================
        # REQUEST ERROR
        # ========================================================

        except requests.exceptions.RequestException as e:

            print(

                f"REQUEST ERROR: {e}",

                flush=True

            )

            return None


        # ========================================================
        # UNEXPECTED ERROR
        # ========================================================

        except Exception as e:

            print(

                f"UNEXPECTED ERROR: {e}",

                flush=True

            )

            return None


    # ============================================================
    # ALL RETRIES FAILED
    # ============================================================

    print(

        "\n REQUEST FAILED AFTER "
        f"{MAX_RETRIES} ATTEMPTS",

        flush=True

    )


    return None


# ================================================================
# BUILD DATAFRAME FOR ONE POINT
# ================================================================

def build_point_dataframe(

    district,

    point_id,

    lat,

    lon,

    data

):

    if data is None:

        return None


    if "hourly" not in data:

        log.warning(

            "%s Point %s: No hourly data returned.",

            district,

            point_id

        )

        return None


    h = data["hourly"]


    times = pd.to_datetime(

        h["time"]

    )


    elevation = data.get(

        "elevation"

    )


    df = pd.DataFrame({

        "District":
            district,

        "PointID":
            point_id,

        "Latitude":
            lat,

        "Longitude":
            lon,

        "Elevation_m":
            elevation,

        "DateTime":
            h["time"],

        "Date":
            times.date,

        "Day":
            times.day,

        "Month":
            times.month,

        "Year":
            times.year,

        "Time":
            times.strftime("%H:%M"),

        "Season": [

            get_season(month)

            for month in times.month

        ],

        # --------------------------------------------------------
        # TEMPERATURE
        # --------------------------------------------------------

        "Temperature_C":
            h.get("temperature_2m"),

        "FeelsLike_C":
            h.get("apparent_temperature"),

        # --------------------------------------------------------
        # HUMIDITY
        # --------------------------------------------------------

        "Humidity_pct":
            h.get("relative_humidity_2m"),

        "DewPoint_C":
            h.get("dew_point_2m"),

        # --------------------------------------------------------
        # PRESSURE
        # --------------------------------------------------------

        "Pressure_hPa":
            h.get("pressure_msl"),

        # --------------------------------------------------------
        # WIND
        # --------------------------------------------------------

        "WindSpeed_kmh":
            h.get("wind_speed_10m"),

        "WindDirection_deg":
            h.get("wind_direction_10m"),

        "WindDirection_compass": [

            deg_to_compass(x)

            for x in h.get(

                "wind_direction_10m",

                []

            )

        ],

        "WindGust_kmh":
            h.get("wind_gusts_10m"),

        # --------------------------------------------------------
        # RAIN
        # --------------------------------------------------------

        "Rainfall_mm":
            h.get("precipitation"),

        "Rain_mm":
            h.get("rain"),

        # --------------------------------------------------------
        # CLOUD
        # --------------------------------------------------------

        "CloudCover_pct":
            h.get("cloud_cover"),

        # --------------------------------------------------------
        # SOLAR
        # --------------------------------------------------------

        "SolarRadiation_Wm2":
            h.get("shortwave_radiation"),

        "SunshineDuration_s":
            h.get("sunshine_duration"),

        # --------------------------------------------------------
        # VISIBILITY
        # --------------------------------------------------------

        "Visibility_m":
            h.get("visibility"),

    })


    return df


# ================================================================
# BUILD API PARAMETERS
# ================================================================

def build_api_params(

    point_batch,

    start_date,

    end_date

):

    latitudes = [

        p[1]

        for p in point_batch

    ]


    longitudes = [

        p[2]

        for p in point_batch

    ]


    return {

        "latitude":
            ",".join(

                str(x)

                for x in latitudes

            ),

        "longitude":
            ",".join(

                str(x)

                for x in longitudes

            ),

        "start_date":
            start_date,

        "end_date":
            end_date,

        "hourly":
            ",".join(

                HOURLY_VARS

            ),

        "timezone":
            "Asia/Colombo",

        "wind_speed_unit":
            "kmh",

        "temperature_unit":
            "celsius",

        "precipitation_unit":
            "mm",

    }


# ================================================================
# SCRAPE ONE BATCH
# ================================================================

def scrape_point_batch(

    district,

    point_batch,

    start_date,

    end_date,

    batch_number

):

    if not point_batch:

        return []


    point_ids = [

        p[0]

        for p in point_batch

    ]


    latitudes = [

        p[1]

        for p in point_batch

    ]


    longitudes = [

        p[2]

        for p in point_batch

    ]


    params = build_api_params(

        point_batch,

        start_date,

        end_date

    )


    data = fetch_with_retry(

        params,

        district=district,

        batch_number=batch_number,

        start_date=start_date,

        end_date=end_date

    )


    if data is None:

        return None


    # ------------------------------------------------------------
    # SINGLE LOCATION RESPONSE
    # ------------------------------------------------------------

    if isinstance(

        data,

        dict

    ):

        data = [

            data

        ]


    results = []


    # ------------------------------------------------------------
    # PROCESS EACH LOCATION
    # ------------------------------------------------------------

    for i, location_data in enumerate(

        data

    ):

        if i >= len(point_batch):

            break


        try:

            df = build_point_dataframe(

                district=district,

                point_id=point_ids[i],

                lat=latitudes[i],

                lon=longitudes[i],

                data=location_data

            )


            if df is not None:

                results.append(

                    df

                )


        except Exception as e:

            log.warning(

                "Processing error | "
                "%s | Point %s | %s",

                district,

                point_ids[i],

                e

            )


    return results


# ================================================================
# SAVE DATA
# ================================================================

def save_district_data(

    district,

    frames

):

    if not frames:

        return


    out_path = os.path.join(

        OUTPUT_DIR,

        f"{safe_filename(district)}.csv"

    )


    new_df = pd.concat(

        frames,

        ignore_index=True

    )


    # ------------------------------------------------------------
    # REMOVE DUPLICATES
    # ------------------------------------------------------------

    new_df = new_df.drop_duplicates(

        subset=[

            "PointID",

            "DateTime"

        ]

    )


    # ------------------------------------------------------------
    # EXISTING CSV
    # ------------------------------------------------------------

    if os.path.exists(

        out_path

    ):

        try:

            existing_df = pd.read_csv(

                out_path

            )


            combined_df = pd.concat(

                [

                    existing_df,

                    new_df

                ],

                ignore_index=True

            )


            combined_df = combined_df.drop_duplicates(

                subset=[

                    "PointID",

                    "DateTime"

                ]

            )


            combined_df = combined_df.sort_values(

                [

                    "PointID",

                    "DateTime"

                ]

            )


            combined_df.to_csv(

                out_path,

                index=False

            )


            print(

                f"Updated: "
                f"{out_path} | "
                f"{len(combined_df):,} total rows",

                flush=True

            )


        except Exception as e:

            print(

                f"Could not update "
                f"{out_path}: {e}",

                flush=True

            )


    # ------------------------------------------------------------
    # NEW CSV
    # ------------------------------------------------------------

    else:

        new_df = new_df.sort_values(

            [

                "PointID",

                "DateTime"

            ]

        )


        new_df.to_csv(

            out_path,

            index=False

        )


        print(

            f"Created: "
            f"{out_path} | "
            f"{len(new_df):,} rows",

            flush=True

        )


# ================================================================
# CREATE POINT BATCHES
# ================================================================

def create_batches(

    point_records,

    batch_size

):

    return [

        point_records[i:i + batch_size]

        for i in range(

            0,

            len(point_records),

            batch_size

        )

    ]


# ================================================================
# MAIN
# ================================================================

def main():

    global HOURLY_QUOTA_EXHAUSTED


    # Reset quota state whenever main() starts
    HOURLY_QUOTA_EXHAUSTED = False


    print(

        "\n"
        + "=" * 70

    )


    print(

        "HOURLY WEATHER SCRAPER STARTING"

    )


    print(

        "=" * 70

    )


    print(

        f"Year range: "
        f"{START_YEAR} → {END_YEAR}"

    )


    print(

        f"Batch size: "
        f"{BATCH_SIZE}"

    )


    print(

        f"Success delay: "
        f"{REQUEST_DELAY} seconds"

    )


    print(

        f"Temporary 429 retries: "
        f"{MAX_RETRIES}"

    )


    print(

        f"Base retry delay: "
        f"{BASE_SLEEP} seconds"

    )


    print(

        "=" * 70

    )


    # ============================================================
    # LOAD DISTRICT GRIDS
    # ============================================================

    district_grids = (

        load_district_point_grids()

    )


    print(

        f"\n Districts loaded: "
        f"{len(district_grids)}",

        flush=True

    )


    # ============================================================
    # PROCESS DISTRICTS
    # ============================================================

    for district, points in district_grids.items():

        # --------------------------------------------------------
        # STOP IF HOURLY QUOTA WAS EXHAUSTED
        # --------------------------------------------------------

        if HOURLY_QUOTA_EXHAUSTED:

            print(

                "\n"
                + "=" * 70,

                flush=True

            )


            print(

                "STOPPING SCRAPER",

                flush=True

            )


            print(

                "The API hourly quota has been exhausted.",

                flush=True

            )


            print(

                "Already completed chunks are safely stored "
                "in checkpoints.",

                flush=True

            )


            print(

                "Run the scraper again after the quota resets "
                "to continue automatically.",

                flush=True

            )


            print(

                "=" * 70,

                flush=True

            )


            break


        print(

            "\n"
            + "=" * 70

        )


        print(

            f"DISTRICT: {district}"

        )


        print(

            "=" * 70

        )


        # --------------------------------------------------------
        # CREATE POINT RECORDS
        # --------------------------------------------------------

        point_records = [

            (

                point_id,

                lat,

                lon

            )

            for point_id, (

                lat,

                lon

            )

            in enumerate(

                points,

                start=1

            )

        ]


        print(

            f"Geographic points: "
            f"{len(point_records)}",

            flush=True

        )


        # --------------------------------------------------------
        # CREATE BATCHES
        # --------------------------------------------------------

        batches = create_batches(

            point_records,

            BATCH_SIZE

        )


        print(

            f"API batches: "
            f"{len(batches)}",

            flush=True

        )


        # --------------------------------------------------------
        # LOAD CHECKPOINT
        # --------------------------------------------------------

        completed_chunks = (

            load_completed_chunks(

                district

            )

        )


        print(

            f"Previously completed chunks: "
            f"{len(completed_chunks)}",

            flush=True

        )


        # ========================================================
        # PROCESS BATCHES
        # ========================================================

        for batch_number, point_batch in enumerate(

            batches,

            start=1

        ):

            # ----------------------------------------------------
            # STOP IF QUOTA EXHAUSTED
            # ----------------------------------------------------

            if HOURLY_QUOTA_EXHAUSTED:

                break


            print(

                "\n"
                + "-" * 70

            )


            print(

                f"BATCH "
                f"{batch_number}/{len(batches)} | "

                f"Points "
                f"{point_batch[0][0]}-"
                f"{point_batch[-1][0]}",

                flush=True

            )


            print(

                "-" * 70

            )


            # ====================================================
            # PROCESS YEAR CHUNKS
            # ====================================================

            for start_date, end_date in year_chunks(

                START_YEAR,

                END_YEAR

            ):

                # ------------------------------------------------
                # STOP IF QUOTA EXHAUSTED
                # ------------------------------------------------

                if HOURLY_QUOTA_EXHAUSTED:

                    break


                chunk_key = (

                    batch_number,

                    str(start_date),

                    str(end_date)

                )


                # ------------------------------------------------
                # SKIP COMPLETED
                # ------------------------------------------------

                if chunk_key in completed_chunks:

                    print(

                        f"SKIPPING COMPLETED | "
                        f"Batch {batch_number} | "
                        f"{start_date} → {end_date}",

                        flush=True

                    )

                    continue


                print(

                    f"\n DOWNLOADING | "
                    f"{district} | "
                    f"Batch {batch_number} | "
                    f"{start_date} → {end_date}",

                    flush=True

                )


                # ------------------------------------------------
                # API REQUEST
                # ------------------------------------------------

                frames = scrape_point_batch(

                    district=district,

                    point_batch=point_batch,

                    start_date=start_date,

                    end_date=end_date,

                    batch_number=batch_number

                )


                # ------------------------------------------------
                # REQUEST FAILED
                # ------------------------------------------------

                if frames is None:

                    print(

                        f"\n FAILED | "
                        f"{district} | "
                        f"Batch {batch_number} | "
                        f"{start_date} → {end_date}",

                        flush=True

                    )


                    if HOURLY_QUOTA_EXHAUSTED:

                        record_failed_request(

                            district,

                            batch_number,

                            start_date,

                            end_date,

                            "Hourly API quota exceeded."

                        )


                        break


                    record_failed_request(

                        district,

                        batch_number,

                        start_date,

                        end_date,

                        "API request failed after retries."

                    )


                    continue


                # ------------------------------------------------
                # NO DATA
                # ------------------------------------------------

                if not frames:

                    print(

                        "No data returned.",

                        flush=True

                    )


                    record_failed_request(

                        district,

                        batch_number,

                        start_date,

                        end_date,

                        "No data returned."

                    )


                    continue


                # ------------------------------------------------
                # SAVE IMMEDIATELY
                # ------------------------------------------------

                save_district_data(

                    district,

                    frames

                )


                # ------------------------------------------------
                # MARK COMPLETE
                # ------------------------------------------------

                mark_chunk_completed(

                    district,

                    batch_number,

                    start_date,

                    end_date

                )


                completed_chunks.add(

                    chunk_key

                )


                print(

                    f"COMPLETED | "
                    f"{district} | "
                    f"Batch {batch_number} | "
                    f"{start_date} → {end_date}",

                    flush=True

                )


        # ========================================================
        # DISTRICT SUMMARY
        # ========================================================

        out_path = os.path.join(

            OUTPUT_DIR,

            f"{safe_filename(district)}.csv"

        )


        if os.path.exists(

            out_path

        ):

            try:

                df = pd.read_csv(

                    out_path

                )


                print(

                    "\n"
                    + "-" * 70

                )


                print(

                    f"DISTRICT SUMMARY: "
                    f"{district}"

                )


                print(

                    f"Rows: "
                    f"{len(df):,}"

                )


                print(

                    f"Points: "
                    f"{df['PointID'].nunique():,}"

                )


                print(

                    f"Date range: "
                    f"{df['DateTime'].min()} → "
                    f"{df['DateTime'].max()}"

                )


                print(

                    f"File: "
                    f"{out_path}"

                )


                print(

                    "-" * 70

                )


            except Exception as e:

                print(

                    f"Could not summarize "
                    f"{district}: {e}",

                    flush=True

                )


        # --------------------------------------------------------
        # STOP AFTER QUOTA EXHAUSTION
        # --------------------------------------------------------

        if HOURLY_QUOTA_EXHAUSTED:

            break


    # ============================================================
    # FINAL STATUS
    # ============================================================

    print(

        "\n"
        + "=" * 70

    )


    if HOURLY_QUOTA_EXHAUSTED:

        print(

            "SCRAPER PAUSED — HOURLY API QUOTA EXHAUSTED"

        )


        print(

            "Completed chunks remain safely checkpointed."

        )


        print(

            "Run this cell again after the quota resets."

        )


    else:

        print(

            "ALL DISTRICTS PROCESSED"

        )


    print(

        "=" * 70

    )


    print(

        f"Output directory: "
        f"{OUTPUT_DIR}"

    )


    print(

        f"Checkpoint directory: "
        f"{CHECKPOINT_DIR}"

    )


    print(

        f"Failed requests file: "
        f"{FAILED_LOG_PATH}"

    )


    print(

        "=" * 70

    )


# ================================================================
# RUN
# ================================================================

if __name__ == "__main__":

    main()


HOURLY WEATHER SCRAPER STARTING
Year range: 2020 → 2025
Batch size: 10
Success delay: 20.0 seconds
Temporary 429 retries: 3
Base retry delay: 120.0 seconds

 Districts loaded: 14

DISTRICT: Colombo
Geographic points: 11
API batches: 2
Previously completed chunks: 12

----------------------------------------------------------------------
BATCH 1/2 | Points 1-10
----------------------------------------------------------------------
SKIPPING COMPLETED | Batch 1 | 2020-01-01 → 2020-12-31
SKIPPING COMPLETED | Batch 1 | 2021-01-01 → 2021-12-31
SKIPPING COMPLETED | Batch 1 | 2022-01-01 → 2022-12-31
SKIPPING COMPLETED | Batch 1 | 2023-01-01 → 2023-12-31
SKIPPING COMPLETED | Batch 1 | 2024-01-01 → 2024-12-31
SKIPPING COMPLETED | Batch 1 | 2025-01-01 → 2025-12-31

----------------------------------------------------------------------
BATCH 2/2 | Points 11-11
----------------------------------------------------------------------
SKIPPING COMPLETED | Batch 2 | 2020-01-01 → 2020-12-31
SKIPPING COM

In [ ]:
import os
import pandas as pd

colombo_path = os.path.join(
    OUTPUT_DIR,
    "Colombo.csv"
)

print("File:", colombo_path)
print("Exists:", os.path.exists(colombo_path))

if os.path.exists(colombo_path):

    df = pd.read_csv(colombo_path)

    print("\n==============================")
    print("COLOMBO DATA")
    print("==============================")

    print("Rows:", f"{len(df):,}")
    print("Columns:", len(df))
    print("Points:", df["PointID"].nunique())

    print(
        "Date range:",
        df["DateTime"].min(),
        "→",
        df["DateTime"].max()
    )

    print("\nRows per point:")
    print(
        df.groupby("PointID")
          .size()
    )

    print("\nFirst 5 rows:")
    display(df.head())

else:

    print("\n❌ Colombo.csv does not exist.")

File: sl_weather_data/Colombo.csv
Exists: True

COLOMBO DATA
Rows: 578,688
Columns: 578688
Points: 11
Date range: 2020-01-01T00:00 → 2025-12-31T23:00

Rows per point:
PointID
1     52608
2     52608
3     52608
4     52608
5     52608
6     52608
7     52608
8     52608
9     52608
10    52608
11    52608
dtype: int64

First 5 rows:


,District,PointID,Latitude,Longitude,Elevation_m,DateTime,Date,Day,Month,Year,...,WindSpeed_kmh,WindDirection_deg,WindDirection_compass,WindGust_kmh,Rainfall_mm,Rain_mm,CloudCover_pct,SolarRadiation_Wm2,SunshineDuration_s,Visibility_m
0,Colombo,1,6.85991,79.79351,0.0,2020-01-01T00:00,2020-01-01,1,1,2020,...,9.0,157,SSE,22.0,0.1,0.1,43,0.0,0.0,NaN
1,Colombo,1,6.85991,79.79351,0.0,2020-01-01T01:00,2020-01-01,1,1,2020,...,5.5,148,SSE,15.8,0.0,0.0,50,0.0,0.0,NaN
2,Colombo,1,6.85991,79.79351,0.0,2020-01-01T02:00,2020-01-01,1,1,2020,...,4.3,156,SSE,9.7,0.1,0.1,78,0.0,0.0,NaN
3,Colombo,1,6.85991,79.79351,0.0,2020-01-01T03:00,2020-01-01,1,1,2020,...,3.9,146,SE,8.6,0.1,0.1,92,0.0,0.0,NaN
4,Colombo,1,6.85991,79.79351,0.0,2020-01-01T04:00,2020-01-01,1,1,2020,...,3.6,96,E,7.2,0.0,0.0,90,0.0,0.0,NaN


In [ ]:
# ================================================================
# GOOGLE COLAB — DOWNLOAD COMPLETED DISTRICT DATASETS
# ================================================================

import os
import shutil
import zipfile
import pandas as pd
from google.colab import files


# ================================================================
# DISTRICTS TO DOWNLOAD
# ================================================================

COMPLETED_DISTRICTS = [
    "Colombo",
    "Gampaha"
]


# ================================================================
# CHECK OUTPUT_DIR
# ================================================================
#
# OUTPUT_DIR should already be defined by your scraper.
#
# Example:
# OUTPUT_DIR = "/content/weather_data"
#
# ================================================================

if "OUTPUT_DIR" not in globals():

    raise NameError(
        """
OUTPUT_DIR is not defined.

Please run the previous scraper cells first,
where OUTPUT_DIR is defined.

For example:

OUTPUT_DIR = "/content/weather_data"
        """
    )


print("=" * 80)
print("📥 DOWNLOADING COMPLETED WEATHER DATASETS")
print("=" * 80)

print(
    f"\n📁 Scraper output directory:\n{OUTPUT_DIR}"
)

print(
    f"\n📁 Output directory exists: "
    f"{os.path.exists(OUTPUT_DIR)}"
)


# ================================================================
# COLAB DOWNLOAD DIRECTORY
# ================================================================

DOWNLOAD_DIR = "/content/downloaded_datasets"

os.makedirs(
    DOWNLOAD_DIR,
    exist_ok=True
)


# ================================================================
# STORE SUCCESSFUL DOWNLOADS
# ================================================================

downloaded_files = []


# ================================================================
# PROCESS EACH DISTRICT
# ================================================================

for district in COMPLETED_DISTRICTS:

    print("\n")
    print("=" * 80)
    print(f"🏙️ {district.upper()}")
    print("=" * 80)


    # ------------------------------------------------------------
    # SOURCE FILE
    # ------------------------------------------------------------

    source_path = os.path.join(
        OUTPUT_DIR,
        f"{district}.csv"
    )


    print(
        f"\n🔎 Checking:\n{source_path}"
    )


    # ------------------------------------------------------------
    # CHECK FILE EXISTS
    # ------------------------------------------------------------

    if not os.path.exists(source_path):

        print(
            f"\n❌ {district}.csv NOT FOUND"
        )

        continue


    # ------------------------------------------------------------
    # FILE SIZE
    # ------------------------------------------------------------

    file_size_mb = (
        os.path.getsize(source_path)
        /
        (1024 ** 2)
    )


    print(
        f"✅ File found"
    )

    print(
        f"📦 File size: "
        f"{file_size_mb:.2f} MB"
    )


    # ------------------------------------------------------------
    # READ DATASET
    # ------------------------------------------------------------

    try:

        df = pd.read_csv(
            source_path
        )

    except Exception as e:

        print(
            f"\n❌ Could not read "
            f"{district}.csv"
        )

        print(
            f"Error: {e}"
        )

        continue


    # ============================================================
    # DATASET INFORMATION
    # ============================================================

    print("\n📊 DATASET INFORMATION")
    print("-" * 50)


    print(
        f"Rows: "
        f"{len(df):,}"
    )


    print(
        f"Columns: "
        f"{len(df.columns):,}"
    )


    # ------------------------------------------------------------
    # POINT INFORMATION
    # ------------------------------------------------------------

    if "PointID" in df.columns:

        print(
            f"Geographic points: "
            f"{df['PointID'].nunique():,}"
        )

        print("\nPoints:")

        print(
            sorted(
                df["PointID"]
                .dropna()
                .unique()
                .tolist()
            )
        )


    # ------------------------------------------------------------
    # DATE INFORMATION
    # ------------------------------------------------------------

    if "DateTime" in df.columns:

        print(
            f"\nDate range:"
        )

        print(
            f"{df['DateTime'].min()} "
            f"→ "
            f"{df['DateTime'].max()}"
        )


    # ------------------------------------------------------------
    # COLUMN NAMES
    # ------------------------------------------------------------

    print("\n📋 Columns:")

    for i, column in enumerate(
        df.columns,
        start=1
    ):

        print(
            f"{i:2}. {column}"
        )


    # ============================================================
    # CHECK FOR DUPLICATES
    # ============================================================

    if (
        "PointID" in df.columns
        and
        "DateTime" in df.columns
    ):

        duplicate_count = df.duplicated(
            subset=[
                "PointID",
                "DateTime"
            ]
        ).sum()


        print(
            f"\n🔍 Duplicate "
            f"(PointID + DateTime) rows: "
            f"{duplicate_count:,}"
        )


    # ============================================================
    # SHOW FIRST 5 ROWS
    # ============================================================

    print("\n👀 FIRST 5 ROWS")

    display(
        df.head()
    )


    # ============================================================
    # COPY FILE TO COLAB DOWNLOAD DIRECTORY
    # ============================================================

    destination_path = os.path.join(
        DOWNLOAD_DIR,
        f"{district}.csv"
    )


    shutil.copy2(
        source_path,
        destination_path
    )


    downloaded_files.append(
        destination_path
    )


    print(
        f"\n✅ Prepared for download:"
    )

    print(
        destination_path
    )


# ================================================================
# CHECK WHETHER FILES WERE FOUND
# ================================================================

print("\n")
print("=" * 80)
print("📦 DOWNLOAD SUMMARY")
print("=" * 80)


if not downloaded_files:

    print(
        "\n❌ No completed datasets were found."
    )

    print(
        "\nCheck your OUTPUT_DIR:"
    )

    print(
        OUTPUT_DIR
    )


else:

    for path in downloaded_files:

        size_mb = (
            os.path.getsize(path)
            /
            (1024 ** 2)
        )

        print(
            f"✅ {os.path.basename(path)} "
            f"({size_mb:.2f} MB)"
        )


# ================================================================
# CREATE ZIP FILE
# ================================================================

if downloaded_files:

    zip_path = "/content/Completed_Weather_Datasets.zip"


    print("\n")
    print("=" * 80)
    print("🗜️ CREATING ZIP FILE")
    print("=" * 80)


    with zipfile.ZipFile(
        zip_path,
        "w",
        compression=zipfile.ZIP_DEFLATED
    ) as zip_file:

        for file_path in downloaded_files:

            zip_file.write(
                file_path,
                arcname=os.path.basename(
                    file_path
                )
            )


    zip_size_mb = (
        os.path.getsize(zip_path)
        /
        (1024 ** 2)
    )


    print(
        f"\n✅ ZIP created:"
    )

    print(
        zip_path
    )

    print(
        f"📦 ZIP size: "
        f"{zip_size_mb:.2f} MB"
    )


# ================================================================
# AUTOMATIC DOWNLOAD
# ================================================================
#
# Colab will trigger browser downloads.
#
# ================================================================

print("\n")
print("=" * 80)
print("⬇️ STARTING DOWNLOADS")
print("=" * 80)


if downloaded_files:

    # ------------------------------------------------------------
    # Download individual CSV files
    # ------------------------------------------------------------

    for file_path in downloaded_files:

        print(
            f"\n⬇️ Downloading "
            f"{os.path.basename(file_path)}..."
        )

        files.download(
            file_path
        )


    # ------------------------------------------------------------
    # Download ZIP as well
    # ------------------------------------------------------------

    if os.path.exists(zip_path):

        print(
            "\n⬇️ Downloading ZIP containing all datasets..."
        )

        files.download(
            zip_path
        )


print("\n")
print("=" * 80)
print("🎉 DOWNLOAD PROCESS COMPLETED")
print("=" * 80)

📥 DOWNLOADING COMPLETED WEATHER DATASETS

📁 Scraper output directory:
sl_weather_data

📁 Output directory exists: True


🏙️ COLOMBO

🔎 Checking:
sl_weather_data/Colombo.csv
✅ File found
📦 File size: 86.73 MB

📊 DATASET INFORMATION
--------------------------------------------------
Rows: 578,688
Columns: 27
Geographic points: 11

Points:
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]

Date range:
2020-01-01T00:00 → 2025-12-31T23:00

📋 Columns:
 1. District
 2. PointID
 3. Latitude
 4. Longitude
 5. Elevation_m
 6. DateTime
 7. Date
 8. Day
 9. Month
10. Year
11. Time
12. Season
13. Temperature_C
14. FeelsLike_C
15. Humidity_pct
16. DewPoint_C
17. Pressure_hPa
18. WindSpeed_kmh
19. WindDirection_deg
20. WindDirection_compass
21. WindGust_kmh
22. Rainfall_mm
23. Rain_mm
24. CloudCover_pct
25. SolarRadiation_Wm2
26. SunshineDuration_s
27. Visibility_m

🔍 Duplicate (PointID + DateTime) rows: 0

👀 FIRST 5 ROWS


,District,PointID,Latitude,Longitude,Elevation_m,DateTime,Date,Day,Month,Year,...,WindSpeed_kmh,WindDirection_deg,WindDirection_compass,WindGust_kmh,Rainfall_mm,Rain_mm,CloudCover_pct,SolarRadiation_Wm2,SunshineDuration_s,Visibility_m
0,Colombo,1,6.85991,79.79351,0.0,2020-01-01T00:00,2020-01-01,1,1,2020,...,9.0,157,SSE,22.0,0.1,0.1,43,0.0,0.0,NaN
1,Colombo,1,6.85991,79.79351,0.0,2020-01-01T01:00,2020-01-01,1,1,2020,...,5.5,148,SSE,15.8,0.0,0.0,50,0.0,0.0,NaN
2,Colombo,1,6.85991,79.79351,0.0,2020-01-01T02:00,2020-01-01,1,1,2020,...,4.3,156,SSE,9.7,0.1,0.1,78,0.0,0.0,NaN
3,Colombo,1,6.85991,79.79351,0.0,2020-01-01T03:00,2020-01-01,1,1,2020,...,3.9,146,SE,8.6,0.1,0.1,92,0.0,0.0,NaN
4,Colombo,1,6.85991,79.79351,0.0,2020-01-01T04:00,2020-01-01,1,1,2020,...,3.6,96,E,7.2,0.0,0.0,90,0.0,0.0,NaN



✅ Prepared for download:
/content/downloaded_datasets/Colombo.csv


🏙️ GAMPAHA

🔎 Checking:
sl_weather_data/Gampaha.csv
✅ File found
📦 File size: 135.02 MB

📊 DATASET INFORMATION
--------------------------------------------------
Rows: 894,336
Columns: 27
Geographic points: 17

Points:
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]

Date range:
2020-01-01T00:00 → 2025-12-31T23:00

📋 Columns:
 1. District
 2. PointID
 3. Latitude
 4. Longitude
 5. Elevation_m
 6. DateTime
 7. Date
 8. Day
 9. Month
10. Year
11. Time
12. Season
13. Temperature_C
14. FeelsLike_C
15. Humidity_pct
16. DewPoint_C
17. Pressure_hPa
18. WindSpeed_kmh
19. WindDirection_deg
20. WindDirection_compass
21. WindGust_kmh
22. Rainfall_mm
23. Rain_mm
24. CloudCover_pct
25. SolarRadiation_Wm2
26. SunshineDuration_s
27. Visibility_m

🔍 Duplicate (PointID + DateTime) rows: 0

👀 FIRST 5 ROWS


,District,PointID,Latitude,Longitude,Elevation_m,DateTime,Date,Day,Month,Year,...,WindSpeed_kmh,WindDirection_deg,WindDirection_compass,WindGust_kmh,Rainfall_mm,Rain_mm,CloudCover_pct,SolarRadiation_Wm2,SunshineDuration_s,Visibility_m
0,Gampaha,1,6.97796,79.88538,4.0,2020-01-01T00:00,2020-01-01,1,1,2020,...,8.1,148,SSE,20.9,0.1,0.1,34,0.0,0.0,NaN
1,Gampaha,1,6.97796,79.88538,4.0,2020-01-01T01:00,2020-01-01,1,1,2020,...,6.4,133,SE,14.4,0.0,0.0,50,0.0,0.0,NaN
2,Gampaha,1,6.97796,79.88538,4.0,2020-01-01T02:00,2020-01-01,1,1,2020,...,4.8,138,SE,11.9,0.0,0.0,81,0.0,0.0,NaN
3,Gampaha,1,6.97796,79.88538,4.0,2020-01-01T03:00,2020-01-01,1,1,2020,...,3.7,119,ESE,9.0,0.0,0.0,92,0.0,0.0,NaN
4,Gampaha,1,6.97796,79.88538,4.0,2020-01-01T04:00,2020-01-01,1,1,2020,...,4.4,81,E,8.6,0.0,0.0,91,0.0,0.0,NaN



✅ Prepared for download:
/content/downloaded_datasets/Gampaha.csv


📦 DOWNLOAD SUMMARY
✅ Colombo.csv (86.73 MB)
✅ Gampaha.csv (135.02 MB)


🗜️ CREATING ZIP FILE

✅ ZIP created:
/content/Completed_Weather_Datasets.zip
📦 ZIP size: 35.42 MB


⬇️ STARTING DOWNLOADS

⬇️ Downloading Colombo.csv...


...

<IPython.core.display.Javascript object>


⬇️ Downloading ZIP containing all datasets...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>



🎉 DOWNLOAD PROCESS COMPLETED
